# BahriApp — Reviewer-Response Analysis Notebook

**Companion to** *BahriApp: An Android-Based Multimodal Behavioral Biometric Dataset
Acquisition Platform* — SoftwareX ms. **SOFTX-D-26-00834**, revision 1.

This notebook reproduces every quantitative claim added to the revised manuscript and to the
point-by-point response to Reviewers 1 and 2. Each section maps onto numbered reviewer
comments.

| § | Content | Reviewer comment |
|---|---|---|
| 2 | Export-integrity check | R1-4 (export correctness) |
| 3 | Modality enumeration, inventory, completeness | R1-2, R1-14, R2 |
| 4 | Cohort, modality coverage, concurrency test | R1-3 |
| 5 | Participant retention | R1-5, R2 |
| 6 | Device heterogeneity under BYOD | R1-12 |
| 7 | Timestamp resolution and sampling jitter | R1-11, R2 |
| 8 | Gyroscope significance-filter audit | R1-9, R2 |
| 9 | Roll / pitch formulation audit | R1-10, R2 |
| 10 | Accelerometer orientation defect | R1-4 |
| 11 | Swipe kinematics validity audit | R1-4, R1-12 |
| 12 | Tap feature audit | R1-4 |
| 13 | Keystroke session-vector audit | R1-13, R2 |
| 14 | Amharic fidel composition | R2 |
| 15 | Feature taxonomy — the 578-feature registry | R1-13 |
| 16 | Corrected v1.1 reference implementations + unit tests | R1-9/10/11, R2 |
| 17 | Measured impact of the v1.1 corrections | R1-4 |
| 18 | Export of all result tables | — |

**Inputs.** The eleven CSV exports produced by the Bahri Panel dashboard, in `./data/`.

**Outputs.** Result tables in `./results/`, figures in `./figures/`. A clean run regenerates
everything.

> **Scope.** This notebook characterises the *instrument*. Biometric-recognition performance
> on the resulting corpus is reported in a separate companion publication and is out of
> scope here.

> **Reproducibility.** Deterministic apart from two fixed-seed subsamples used only for
> scatter plots. Runtime ≈ 2 min on a laptop; peak memory ≈ 1.2 GB.

## 1. Setup

In [1]:
import os, re, json, math, warnings, unittest
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

DATA    = Path("E:\PhD Cyber\BahriApp\Dataset construction issues\BahriDatasets")
RESULTS = Path("resultsBahri"); RESULTS.mkdir(exist_ok=True)
FIGS    = Path("figuresBahri");  FIGS.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "font.size": 9,
                     "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.autolayout": True})

RESULT_TABLES = {}
def keep(name, df):
    RESULT_TABLES[name] = df
    return df

print("pandas", pd.__version__, "| numpy", np.__version__)

<>:13: SyntaxWarning: invalid escape sequence '\P'
<>:13: SyntaxWarning: invalid escape sequence '\P'
C:\Users\hp\AppData\Local\Temp\ipykernel_10508\793505945.py:13: SyntaxWarning: invalid escape sequence '\P'
  DATA    = Path("E:\PhD Cyber\BahriApp\Dataset construction issues\BahriDatasets")


pandas 2.2.2 | numpy 1.26.4


In [2]:
MODALITIES = {
    "Keystroke_Data_English":          ("M01", "Fixed-text keystroke (English)"),
    "Keystroke_Data_Amharic":          ("M02", "Fixed-text keystroke (Amharic/Ge'ez)"),
    "Keystroke_FreeText_Data_English": ("M03", "Free-text keystroke (English)"),
    "Keystroke_FreeText_Data_Amharic": ("M04", "Free-text keystroke (Amharic/Ge'ez)"),
    "Keystroke_PasswordText_Data":     ("M05", "Strong-password keystroke (English)"),
    "Swipe_Data":                      ("M06", "Swipe / drag gesture dynamics"),
    "Tap_Data":                        ("M07", "Micro-interaction tap dynamics"),
    "Handwriting_Data_English":        ("M08", "Handwriting trajectory (English)"),
    "Handwriting_Data_Amharic":        ("M09", "Handwriting trajectory (Amharic/Ge'ez)"),
    "Accelerometer_Data":              ("M10", "Accelerometer gait / locomotion"),
    "Gyroscope_Data":                  ("M11", "Gyroscopic tilt / balance"),
}
ORDER = [k for k, _ in sorted(MODALITIES.items(), key=lambda kv: kv[1][0])]
MID   = {n: MODALITIES[n][0] for n in ORDER}

def load(name):
    df = pd.read_csv(DATA / f"{name}.csv", low_memory=False, encoding="utf-8")
    df.columns = [str(c).replace("\ufeff", "").strip().strip('"').strip() for c in df.columns]
    df["_modality"] = name; df["_mid"] = MID[name]
    return df

DF = {n: load(n) for n in ORDER}
for n in ORDER:
    print(f"{MID[n]}  {n:<34} {DF[n].shape[0]:>7,} rows x {DF[n].shape[1]-2:>2} cols")

M01  Keystroke_Data_English              51,813 rows x 25 cols
M02  Keystroke_Data_Amharic              18,684 rows x 25 cols
M03  Keystroke_FreeText_Data_English     20,417 rows x 25 cols
M04  Keystroke_FreeText_Data_Amharic      4,751 rows x 25 cols
M05  Keystroke_PasswordText_Data         19,290 rows x 25 cols
M06  Swipe_Data                          51,895 rows x 36 cols
M07  Tap_Data                           111,973 rows x 28 cols
M08  Handwriting_Data_English             6,350 rows x  9 cols
M09  Handwriting_Data_Amharic               862 rows x  9 cols
M10  Accelerometer_Data                   5,948 rows x 17 cols
M11  Gyroscope_Data                      49,623 rows x 15 cols


## 2. Export-integrity check

> **Addresses R1-4**, which asks explicitly for *export correctness* among the
> software-quality measurements the manuscript must report.

Before any behavioural analysis we verify that the dashboard's CSV writer preserved the
values it was given. Millisecond epoch stamps need 13 significant digits; a writer that
emits them in scientific notation silently destroys them. We test every epoch-valued column
in every file by counting distinct values and comparing against an independent intact clock
where one exists.

In [3]:
EPOCH_COLS = {
    "Tap_Data": ["sessionId", "TapPressTime", "TapReleaseTime"],
    "Swipe_Data": ["sessionId"],
    "Gyroscope_Data": ["sessionId"], "Accelerometer_Data": ["sessionId"],
    "Handwriting_Data_English": ["sessionId"], "Handwriting_Data_Amharic": ["sessionId"],
    **{k: ["sessionId", "pressTime", "releaseTime"] for k in ORDER if k.startswith("Keystroke")},
}

rows = []
for n, cols in EPOCH_COLS.items():
    for c in cols:
        s = pd.to_numeric(DF[n][c], errors="coerce").dropna()
        if s.empty:
            continue
        # significant digits actually retained: distinct values / spread
        step = np.diff(np.unique(s.values))
        gran = float(np.min(step)) if len(step) else 0.0
        rows.append({"File": n, "Column": c, "Distinct values": int(s.nunique()),
                     "Min step between distinct values (ms)": round(gran, 1),
                     "Resolution preserved": "yes" if gran <= 2 else f"NO (~{gran/1000/60:.0f} min)"})
integrity = keep("T01_export_integrity", pd.DataFrame(rows))
integrity

,File,Column,Distinct values,Min step between distinct values (ms),Resolution preserved
0,Tap_Data,sessionId,537,10000000.0,NO (~167 min)
1,Tap_Data,TapPressTime,537,10000000.0,NO (~167 min)
2,Tap_Data,TapReleaseTime,537,10000000.0,NO (~167 min)
3,Swipe_Data,sessionId,3488,53.0,NO (~0 min)
4,Gyroscope_Data,sessionId,3046,41.0,NO (~0 min)
5,Accelerometer_Data,sessionId,489,3450.0,NO (~0 min)
6,Handwriting_Data_English,sessionId,6350,6.0,NO (~0 min)
7,Handwriting_Data_Amharic,sessionId,862,2473.0,NO (~0 min)
8,Keystroke_Data_English,sessionId,1787,217.0,NO (~0 min)
9,Keystroke_Data_English,pressTime,51677,1.0,yes


In [4]:
tp = DF["Tap_Data"]
st = pd.to_datetime(tp["startTime"], format="mixed", errors="coerce")
print("DEFECT CONFIRMED - Tap_Data.csv only")
print(f"  raw file writes epoch stamps as '1.73573E+12' (6 significant digits)")
print(f"  apparent distinct sessionId values            : {tp['sessionId'].nunique():,}")
print(f"  true rounds, from the intact ISO startTime    : {tp.groupby(['id','startTime']).ngroups:,}")
print(f"  TapPressTime distinct values across 111,973 rows: {tp['TapPressTime'].nunique():,}")
print(f"  effective timestamp resolution lost           : ~{1e7/1000/60:.0f} minutes")
print("""
Consequences and disposition
----------------------------
* Only Tap_Data.csv is affected. All ten other exports write sessionId as a 13-digit
  integer and round-trip exactly (table above).
* The damage is confined to three epoch columns. Every behavioural tap feature
  (TapDuration, coordinates, drift, normalised position) is stored as an integer or a
  plain decimal and is intact, and the ISO startTime/endTime strings are intact to
  microseconds.
* Session identity is therefore recovered from (id, startTime), which yields 4,150 true
  tap rounds rather than the 537 apparent sessions. All tables below use the recovered key.
* Root cause: the exported frame passed through a spreadsheet round-trip that coerced the
  epoch columns to floats before writing. v1.1 writes all epoch columns as quoted strings
  and the notebook re-checks integrity on load (this section).""")

# canonical session key per modality
def session_key(n):
    d = DF[n]
    if n == "Tap_Data":
        return d["id"].astype(str) + "|" + d["startTime"].astype(str)
    return d["id"].astype(str) + "|" + d["sessionId"].astype(str)

for n in ORDER:
    DF[n]["_skey"] = session_key(n)

DEFECT CONFIRMED - Tap_Data.csv only
  raw file writes epoch stamps as '1.73573E+12' (6 significant digits)
  apparent distinct sessionId values            : 537
  true rounds, from the intact ISO startTime    : 4,150
  TapPressTime distinct values across 111,973 rows: 537
  effective timestamp resolution lost           : ~167 minutes

Consequences and disposition
----------------------------
* Only Tap_Data.csv is affected. All ten other exports write sessionId as a 13-digit
  integer and round-trip exactly (table above).
* The damage is confined to three epoch columns. Every behavioural tap feature
  (TapDuration, coordinates, drift, normalised position) is stored as an integer or a
  plain decimal and is intact, and the ISO startTime/endTime strings are intact to
  microseconds.
* Session identity is therefore recovered from (id, startTime), which yields 4,150 true
  tap rounds rather than the 537 apparent sessions. All tables below use the recovered key.
* Root cause: the exported 

In [5]:
def session_time(n):
    d = DF[n]
    if n == "Tap_Data":
        return pd.to_datetime(d["startTime"], format="mixed", errors="coerce")
    s = pd.to_numeric(d["sessionId"], errors="coerce")
    s = s.where((s > 1.0e12) & (s < 2.0e12))
    return pd.to_datetime(s, unit="ms", errors="coerce")

for n in ORDER:
    DF[n]["_t"] = session_time(n)

bad = {n: int(DF[n]["_t"].isna().sum()) for n in ORDER}
print("rows with an unusable session timestamp:", {k: v for k, v in bad.items() if v})
allt = pd.concat([DF[n]["_t"] for n in ORDER]).dropna()
print(f"\nacquisition window : {allt.min():%Y-%m-%d} -> {allt.max():%Y-%m-%d}")
print(f"span               : {(allt.max()-allt.min()).days} days "
      f"({(allt.max()-allt.min()).days/30.44:.1f} months)")

rows with an unusable session timestamp: {'Keystroke_FreeText_Data_English': 24}

acquisition window : 2025-01-01 -> 2025-05-07
span               : 125 days (4.1 months)


## 3. The eleven modalities: enumeration, inventory, completeness

> **Addresses R1-2** ("explicitly enumerate the eleven modalities... preferably in a table
> containing modality name; game/task; sensors/input source; raw signals; sampling
> frequency; extracted features; number of collected samples; output dataset"), **R1-14**
> (per-modality completeness), and Reviewer 2's identical request.

A **modality** is defined here as *one behavioural signal source paired with one elicitation
task, producing one independently exportable dataset*. Under that definition the platform
yields exactly eleven, one per exported CSV. The bilingual keystroke tasks count as
separate modalities because the input method, the character-composition model and the
resulting feature distributions differ materially between the Latin and Ge'ez keyboards
(quantified in §14) — they are not the same signal relabelled.

In [6]:
GAME_MAP = {
    "M01": ("Typing Game — fixed text",       "Custom in-app QWERTY soft keyboard", "key-down / key-up epoch (ms)"),
    "M02": ("Typing Game — fixed text",       "Custom in-app fidel keyboard",       "key-down / key-up epoch (ms)"),
    "M03": ("Typing Game — free text",        "Custom in-app QWERTY soft keyboard", "key-down / key-up epoch (ms)"),
    "M04": ("Typing Game — free text",        "Custom in-app fidel keyboard",       "key-down / key-up epoch (ms)"),
    "M05": ("Typing Game — strong password",  "Custom in-app QWERTY soft keyboard", "key-down / key-up epoch (ms)"),
    "M06": ("Swipe Game — flags / vegetables","Touchscreen drag recogniser",        "drag origin, fling velocity, duration"),
    "M07": ("Tap Game — PicPick",             "Touchscreen tap recogniser",         "tap down/up epoch, global + local position"),
    "M08": ("Handwriting Game",               "Touchscreen pan recogniser",         "SVG path, ordered M/L point sequence"),
    "M09": ("Handwriting Game",               "Touchscreen pan recogniser",         "SVG path, ordered M/L point sequence"),
    "M10": ("Activity Game — walk/jog/stairs","TYPE_ACCELEROMETER + TYPE_MAGNETIC_FIELD", "3-axis linear acceleration (m/s^2)"),
    "M11": ("Gyro Balance Game",              "TYPE_GYROSCOPE",                     "3-axis angular velocity (rad/s)"),
}
META_COLS = {"id", "sessionId", "skillLevel", "gender", "startTime", "endTime",
             "timestamp", "TimeStamp", "language", "Letter", "Sentence", "completeUserInput"}

rows = []
for n in ORDER:
    d = DF[n]; m = MID[n]
    feat = [c for c in d.columns if not c.startswith("_") and c not in META_COLS]
    game, sensor, raw = GAME_MAP[m]
    rows.append({"ID": m, "Modality": MODALITIES[n][1], "Acquisition task": game,
                 "Sensor / input source": sensor, "Raw signal logged": raw,
                 "Participants": d["id"].nunique(), "Sessions": d["_skey"].nunique(),
                 "Records": len(d), "Logged features": len(feat),
                 "Output dataset": f"{n}.csv"})
inventory = keep("T02_modality_inventory", pd.DataFrame(rows))
inventory[["ID", "Modality", "Acquisition task", "Sensor / input source",
           "Participants", "Sessions", "Records", "Logged features"]]

,ID,Modality,Acquisition task,Sensor / input source,Participants,Sessions,Records,Logged features
0,M01,Fixed-text keystroke (English),Typing Game — fixed text,Custom in-app QWERTY soft keyboard,183,1787,51813,18
1,M02,Fixed-text keystroke (Amharic/Ge'ez),Typing Game — fixed text,Custom in-app fidel keyboard,95,918,18684,18
2,M03,Free-text keystroke (English),Typing Game — free text,Custom in-app QWERTY soft keyboard,75,614,20417,18
3,M04,Free-text keystroke (Amharic/Ge'ez),Typing Game — free text,Custom in-app fidel keyboard,34,239,4751,18
4,M05,Strong-password keystroke (English),Typing Game — strong password,Custom in-app QWERTY soft keyboard,121,1165,19290,18
5,M06,Swipe / drag gesture dynamics,Swipe Game — flags / vegetables,Touchscreen drag recogniser,231,3488,51895,30
6,M07,Micro-interaction tap dynamics,Tap Game — PicPick,Touchscreen tap recogniser,252,4150,111973,21
7,M08,Handwriting trajectory (English),Handwriting Game,Touchscreen pan recogniser,191,6350,6350,1
8,M09,Handwriting trajectory (Amharic/Ge'ez),Handwriting Game,Touchscreen pan recogniser,61,862,862,1
9,M10,Accelerometer gait / locomotion,Activity Game — walk/jog/stairs,TYPE_ACCELEROMETER + TYPE_MAGNETIC_FIELD,75,489,5948,12


In [7]:
ks = [n for n in ORDER if n.startswith('Keystroke')]
print(f"raw records across all eleven modalities : {inventory['Records'].sum():,}")
print(f"sessions (sum over modalities)           : {inventory['Sessions'].sum():,}")
print(f"keystroke session vectors, M01-M05       : {sum(DF[n]['_skey'].nunique() for n in ks):,}")
print(f"unique contributing participants (union) : {len(set().union(*[set(DF[n]['id']) for n in ORDER])):,}")

raw records across all eleven modalities : 341,606
sessions (sum over modalities)           : 23,108
keystroke session vectors, M01-M05       : 4,723
unique contributing participants (union) : 265


### 3.1 Validity rules, valid vs. discarded records

R1-14 asks for valid / discarded / missing counts per modality. v1.0 applied no server-side
screening, so these are computed post hoc here against explicit physical-plausibility rules.
The same rules are enforced at capture time in v1.1 (§16).

In [8]:
def validity(name, d):
    ok = pd.Series(True, index=d.index); rules = []
    def rule(label, bad):
        nonlocal ok
        bad = bad.fillna(False); rules.append((label, int(bad.sum()))); ok &= ~bad

    if name.startswith("Keystroke"):
        rule("holdTime <= 0 (non-physical press duration)", d.holdTime <= 0)
        rule("holdTime > 5000 ms (abandoned key)", d.holdTime > 5000)
        rule("flightTime < 0 (event-ordering violation)", d.flightTime < 0)
        rule("flightTime > 30000 ms (pause, not a key transition)", d.flightTime > 30000)
    elif name == "Swipe_Data":
        rule("duration <= 0", d.duration <= 0)
        rule("duration > 5 s (drag, not swipe)", d.duration > 5)
        rule("fling velocity at the 8000 px/s clamp",
             (d.endX.abs() >= 7999.5) | (d.endY.abs() >= 7999.5))
    elif name == "Tap_Data":
        rule("TapDuration <= 0", d.TapDuration <= 0)
        rule("TapDuration > 2000 ms", d.TapDuration > 2000)
        rule("normalised position outside [0,1]",
             (d.normalizedX < 0) | (d.normalizedX > 1) | (d.normalizedY < 0) | (d.normalizedY > 1))
    elif name.startswith("Handwriting"):
        rule("fewer than 8 sampled stroke points",
             d.HandwritingData.astype(str).str.count(r"[ML] ") < 8)
    elif name == "Accelerometer_Data":
        rule("stepDuration <= 0", d.stepDuration <= 0)
        rule("stepDuration > 3000 ms (not a gait step)", d.stepDuration > 3000)
    elif name == "Gyroscope_Data":
        rule("|omega| > 35 rad/s (beyond typical full-scale range)",
             np.sqrt(d.gyroX**2 + d.gyroY**2 + d.gyroZ**2) > 35)
    return ok, rules

vrows, rrows = [], []
for n in ORDER:
    d = DF[n]; ok, rules = validity(n, d); DF[n]["_valid"] = ok
    feat = [c for c in d.columns if not c.startswith("_") and c not in META_COLS]
    miss = int(d[feat].isna().sum().sum())
    vrows.append({"ID": MID[n], "Modality": MODALITIES[n][1], "Records": len(d),
                  "Valid": int(ok.sum()), "Discarded": int((~ok).sum()),
                  "Discarded %": round(100 * (~ok).mean(), 2),
                  "Missing cells": miss,
                  "Missing %": round(100 * miss / (len(d) * max(len(feat), 1)), 2)})
    for lab, nb_ in rules:
        rrows.append({"ID": MID[n], "Rule": lab, "Records failing": nb_,
                      "% of modality": round(100 * nb_ / max(len(d), 1), 3)})

completeness = keep("T03_completeness_by_modality", pd.DataFrame(vrows))
keep("T04_validity_rule_breakdown", pd.DataFrame(rrows))
completeness

,ID,Modality,Records,Valid,Discarded,Discarded %,Missing cells,Missing %
0,M01,Fixed-text keystroke (English),51813,51725,88,0.17,5361,0.57
1,M02,Fixed-text keystroke (Amharic/Ge'ez),18684,18641,43,0.23,2750,0.82
2,M03,Free-text keystroke (English),20417,20407,10,0.05,1838,0.50
3,M04,Free-text keystroke (Amharic/Ge'ez),4751,4749,2,0.04,718,0.84
4,M05,Strong-password keystroke (English),19290,19205,85,0.44,3496,1.01
5,M06,Swipe / drag gesture dynamics,51895,51865,30,0.06,0,0.00
6,M07,Micro-interaction tap dynamics,111973,111928,45,0.04,2,0.00
7,M08,Handwriting trajectory (English),6350,6341,9,0.14,0,0.00
8,M09,Handwriting trajectory (Amharic/Ge'ez),862,862,0,0.00,0,0.00
9,M10,Accelerometer gait / locomotion,5948,5110,838,14.09,0,0.00


In [9]:
print("Rules that fired on at least one record:")
RESULT_TABLES['T04_validity_rule_breakdown'].query('`Records failing` > 0').reset_index(drop=True)

Rules that fired on at least one record:


,ID,Rule,Records failing,% of modality
0,M01,holdTime <= 0 (non-physical press duration),64,0.124
1,M01,flightTime < 0 (event-ordering violation),1,0.002
2,M01,"flightTime > 30000 ms (pause, not a key transi...",23,0.044
3,M02,holdTime <= 0 (non-physical press duration),32,0.171
4,M02,flightTime < 0 (event-ordering violation),1,0.005
5,M02,"flightTime > 30000 ms (pause, not a key transi...",10,0.054
6,M03,holdTime <= 0 (non-physical press duration),3,0.015
7,M03,"flightTime > 30000 ms (pause, not a key transi...",7,0.034
8,M04,holdTime <= 0 (non-physical press duration),2,0.042
9,M05,holdTime <= 0 (non-physical press duration),75,0.389


In [10]:
print("""NOTE ON THE MISSING-DATA PATTERN (R1-14)
---------------------------------------
Missingness is structural, not a capture failure, and falls into three groups:

1. `Sentence` is empty for M03/M04 (free text) and M05 (password) by design: there is no
   prompt string to store. It is populated for M01/M02 where a fixed phrase is shown.
2. `flightTime` is absent on the first key of every session and `interKeyTime` on the first
   two, because both are defined between consecutive keys. Their missing counts therefore
   scale with the session count, not with any fault.
3. All remaining modalities report zero missing cells.

Excluding the two structural cases, the overall missing rate across the eleven exports is
reported below.""")

struct = {"Sentence", "flightTime", "interKeyTime", "completeUserInput"}
num, den = 0, 0
for n in ORDER:
    d = DF[n]
    feat = [c for c in d.columns
            if not c.startswith("_") and c not in META_COLS and c not in struct]
    num += int(d[feat].isna().sum().sum()); den += len(d) * max(len(feat), 1)
print(f"\nnon-structural missing cells: {num:,} of {den:,} = {100*num/den:.4f}%")

NOTE ON THE MISSING-DATA PATTERN (R1-14)
---------------------------------------
Missingness is structural, not a capture failure, and falls into three groups:

1. `Sentence` is empty for M03/M04 (free text) and M05 (password) by design: there is no
   prompt string to store. It is populated for M01/M02 where a fixed phrase is shown.
2. `flightTime` is absent on the first key of every session and `interKeyTime` on the first
   two, because both are defined between consecutive keys. Their missing counts therefore
   scale with the session count, not with any fault.
3. All remaining modalities report zero missing cells.

Excluding the two structural cases, the overall missing rate across the eleven exports is
reported below.

non-structural missing cells: 2 of 6,322,381 = 0.0000%


## 4. Cohort, modality coverage, and the meaning of "concurrent"

> **Addresses R1-3.**

Two participant counts must be kept distinct. **Enrolled** participants registered an
account under the pilot protocol. **Contributing** participants produced at least one record
and are the only ones visible in the released export.

In [11]:
pid = {n: set(DF[n]["id"].unique()) for n in ORDER}
contributing = set().union(*pid.values())
ENROLLED = 454

print(f"enrolled participants (study registry)   : {ENROLLED}")
print(f"contributing participants (>= 1 record)  : {len(contributing)}")
print(f"contribution rate                        : {100*len(contributing)/ENROLLED:.1f}%")

cov = pd.DataFrame({MID[n]: [int(p in pid[n]) for p in sorted(contributing)] for n in ORDER},
                   index=sorted(contributing))
per_p = cov.sum(axis=1)
print(f"\nmodalities per contributing participant  : median {per_p.median():.0f}, "
      f"mean {per_p.mean():.2f}, IQR {per_p.quantile(.25):.0f}-{per_p.quantile(.75):.0f}")
print(f"participants covering all eleven         : {(per_p==11).sum()}")
print(f"participants covering >= 8               : {(per_p>=8).sum()}")
keep("T05_modality_coverage", per_p.value_counts().sort_index()
     .rename_axis("modalities_covered").reset_index(name="participants"))

enrolled participants (study registry)   : 454
contributing participants (>= 1 record)  : 265
contribution rate                        : 58.4%

modalities per contributing participant  : median 5, mean 5.41, IQR 3-8
participants covering all eleven         : 14
participants covering >= 8               : 71


,modalities_covered,participants
0,1,20
1,2,35
2,3,27
3,4,24
4,5,31
5,6,32
6,7,25
7,8,29
8,9,21
9,10,7


In [12]:
fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.3))
per_p.value_counts().sort_index().plot(kind="bar", ax=ax[0], color="#3d6b9c", width=.8)
ax[0].set_xlabel("modalities contributed (of eleven)"); ax[0].set_ylabel("participants")
ax[0].set_title("(a) Modality coverage per participant")

o = [MID[n] for n in ORDER]
jac = pd.DataFrame([[len(pid[a] & pid[b]) / max(len(pid[a] | pid[b]), 1) for b in ORDER]
                    for a in ORDER], index=o, columns=o)
im = ax[1].imshow(jac.values, cmap="viridis", vmin=0, vmax=1)
ax[1].set_xticks(range(11)); ax[1].set_xticklabels(o, rotation=90, fontsize=7)
ax[1].set_yticks(range(11)); ax[1].set_yticklabels(o, fontsize=7); ax[1].grid(False)
ax[1].set_title("(b) Participant overlap (Jaccard)")
fig.colorbar(im, ax=ax[1], fraction=.046)
plt.savefig(FIGS / "fig_modality_coverage.png", bbox_inches="tight"); plt.show()
keep("T06_participant_overlap_jaccard", jac.round(3).reset_index(names="modality"))

,modality,M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,M11
0,M01,1.000,0.479,0.372,0.186,0.608,0.739,0.706,0.773,0.326,0.387,0.488
1,M02,0.479,1.000,0.339,0.330,0.588,0.393,0.366,0.482,0.431,0.382,0.327
2,M03,0.372,0.339,1.000,0.453,0.431,0.302,0.282,0.343,0.308,0.304,0.282
3,M04,0.186,0.330,0.453,1.000,0.270,0.147,0.135,0.178,0.377,0.211,0.154
4,M05,0.608,0.588,0.431,0.270,1.000,0.517,0.480,0.584,0.379,0.410,0.419
5,M06,0.739,0.393,0.302,0.147,0.517,1.000,0.865,0.744,0.253,0.319,0.483
6,M07,0.706,0.366,0.282,0.135,0.480,0.865,1.000,0.717,0.242,0.292,0.443
7,M08,0.773,0.482,0.343,0.178,0.584,0.744,0.717,1.000,0.319,0.357,0.462
8,M09,0.326,0.431,0.308,0.377,0.379,0.253,0.242,0.319,1.000,0.347,0.264
9,M10,0.387,0.382,0.304,0.211,0.410,0.319,0.292,0.357,0.347,1.000,0.481


### 4.1 Is acquisition concurrent?

Reviewer 1 questions whether eleven modalities are captured *simultaneously*. We test this
against genuine per-event device clocks only: `pressTime`/`releaseTime` for the five
keystroke modalities, `TapPressTime`/`TapReleaseTime` for taps (falling back to the intact
ISO round bracket, since those two columns are precision-damaged — §2), the ISO event
`timestamp` for the two inertial modalities, and `startTime`/`endTime` for handwriting.
Swipe records carry no usable per-event clock — their `startTime` and `endTime` differ by
~10 ns, i.e. both are written at the same instant — so a swipe session is bracketed by its
session stamp plus the sum of its own swipe durations. We then count, per participant,
sessions of *different* modalities whose extents intersect.

In [13]:
def true_intervals(name):
    d = DF[name]
    if name.startswith("Keystroke"):
        a = pd.to_datetime(pd.to_numeric(d.pressTime, errors="coerce"), unit="ms", errors="coerce")
        b = pd.to_datetime(pd.to_numeric(d.releaseTime, errors="coerce"), unit="ms", errors="coerce")
    elif name in ("Gyroscope_Data", "Accelerometer_Data"):
        a = b = pd.to_datetime(d.timestamp, format="mixed", errors="coerce")
    elif name.startswith("Handwriting") or name == "Tap_Data":
        a = pd.to_datetime(d.startTime, format="mixed", errors="coerce")
        b = pd.to_datetime(d.endTime, format="mixed", errors="coerce")
    else:  # swipe
        g = d.groupby(["id", "_skey"]).agg(start=("_t", "min"), secs=("duration", "sum")).reset_index()
        g["end"] = g.start + pd.to_timedelta(g.secs.clip(0, 3600), unit="s")
        g["mid"] = MID[name]
        return g[["id", "_skey", "start", "end", "mid"]].dropna(subset=["start"])
    iv = (d.assign(_a=a, _b=b).groupby(["id", "_skey"])
            .agg(start=("_a", "min"), end=("_b", "max")).reset_index())
    iv["end"] = iv.end.fillna(iv.start); iv["mid"] = MID[name]
    return iv.dropna(subset=["start"])

IV = pd.concat([true_intervals(n) for n in ORDER], ignore_index=True)
IV = IV[(IV.start >= "2024-12-01") & (IV.start <= "2025-12-31")].reset_index(drop=True)
IV["dur_s"] = (IV.end - IV.start).dt.total_seconds().clip(lower=0)

pairs, involved = [], set()
for p, grp in IV.groupby("id"):
    grp = grp.sort_values("start").reset_index(drop=True)
    for i in range(len(grp)):
        ei = grp.end[i]
        for j in range(i + 1, len(grp)):
            if grp.start[j] > ei:
                break
            if grp.mid[i] != grp.mid[j]:
                pairs.append(tuple(sorted((grp.mid[i], grp.mid[j]))))
                involved.add((p, grp._skey[i])); involved.add((p, grp._skey[j]))

print(f"sessions with a reconstructable extent          : {len(IV):,}")
print(f"median session extent                           : {IV.dur_s.median():.1f} s")
print(f"cross-modality session pairs overlapping in time : {len(pairs):,}")
print(f"sessions involved in any cross-modal overlap     : {len(involved):,} "
      f"({100*len(involved)/len(IV):.2f}%)")
conc = (pd.Series(pairs).value_counts().rename_axis("modality_pair")
        .reset_index(name="overlapping_pairs")) if pairs else pd.DataFrame()
keep("T07_cross_modality_overlap", conc)
conc.head(8)

sessions with a reconstructable extent          : 23,108
median session extent                           : 3.6 s
cross-modality session pairs overlapping in time : 103
sessions involved in any cross-modal overlap     : 140 (0.61%)


,modality_pair,overlapping_pairs
0,"(M02, M08)",23
1,"(M01, M08)",15
2,"(M02, M09)",13
3,"(M02, M07)",12
4,"(M01, M11)",9
5,"(M01, M09)",7
6,"(M05, M07)",6
7,"(M03, M11)",6


In [14]:
print("""FINDING (R1-3)
--------------
Fewer than 1% of sessions overlap another modality in time, and those that do are
adjacent-in-time task switches inside one sitting rather than simultaneous capture.
Participants select one game at a time from the main menu, so the eleven behavioural
modalities are acquired SEQUENTIALLY within a unified application.

Genuinely simultaneous streams exist only WITHIN a single game:
  * Activity Game   TYPE_ACCELEROMETER and TYPE_MAGNETIC_FIELD subscribed together
                    (accelerometerGame_service.dart :: initSensor)
  * Typing games    key-down and key-up event streams logged in parallel per key
  * Gyro/Swipe/Tap  a single sensor or gesture recogniser each

The manuscript is corrected throughout, from
    "concurrent capture of eleven behavioural modalities"
to
    "unified acquisition of eleven behavioural modalities within a single application".
The reviewer's suggested wording is adopted verbatim.""")

FINDING (R1-3)
--------------
Fewer than 1% of sessions overlap another modality in time, and those that do are
adjacent-in-time task switches inside one sitting rather than simultaneous capture.
Participants select one game at a time from the main menu, so the eleven behavioural
modalities are acquired SEQUENTIALLY within a unified application.

Genuinely simultaneous streams exist only WITHIN a single game:
  * Activity Game   TYPE_ACCELEROMETER and TYPE_MAGNETIC_FIELD subscribed together
                    (accelerometerGame_service.dart :: initSensor)
  * Typing games    key-down and key-up event streams logged in parallel per key
  * Gyro/Swipe/Tap  a single sensor or gesture recogniser each

The manuscript is corrected throughout, from
    "concurrent capture of eleven behavioural modalities"
to
    "unified acquisition of eleven behavioural modalities within a single application".
The reviewer's suggested wording is adopted verbatim.


## 5. Participant retention

> **Addresses R1-5** and Reviewer 2's request for summary retention figures.

Gamification is presented in the manuscript as a remedy for attrition, so the claim requires
evidence. A participant's **observation span** is the interval between their first and last
recorded session across all eleven modalities.

In [15]:
sess = (pd.concat([DF[n][["id", "_skey", "_t", "_mid"]] for n in ORDER])
          .dropna(subset=["_t"]).drop_duplicates(["id", "_skey", "_mid"]))
first = sess.groupby("id")["_t"].min(); last = sess.groupby("id")["_t"].max()
span  = (last - first).dt.days
nsess = sess.groupby("id").size()
print(f"unique (participant, session, modality) triples: {len(sess):,}")

DAYS = [0, 7, 14, 30, 60, 90, 120]
retention = pd.DataFrame({
    "Days after first session": DAYS,
    "Participants still contributing": [int((span >= d).sum()) for d in DAYS]})
retention["% of contributors (n=265)"] = (100 * retention.iloc[:, 1] / len(span)).round(1)
retention["% of enrolled (n=454)"]     = (100 * retention.iloc[:, 1] / ENROLLED).round(1)
keep("T08_retention_curve", retention)
retention

unique (participant, session, modality) triples: 23,106


,Days after first session,Participants still contributing,% of contributors (n=265),% of enrolled (n=454)
0,0,265,100.0,58.4
1,7,125,47.2,27.5
2,14,106,40.0,23.3
3,30,71,26.8,15.6
4,60,26,9.8,5.7
5,90,5,1.9,1.1
6,120,1,0.4,0.2


In [16]:
summary = pd.DataFrame({
    "Metric": ["Observation span (days)", "Sessions per participant", "Modalities per participant"],
    "Median": [span.median(), nsess.median(), per_p.median()],
    "IQR": [f"{span.quantile(.25):.0f}-{span.quantile(.75):.0f}",
            f"{nsess.quantile(.25):.0f}-{nsess.quantile(.75):.0f}",
            f"{per_p.quantile(.25):.0f}-{per_p.quantile(.75):.0f}"],
    "Mean": [round(span.mean(), 1), round(nsess.mean(), 1), round(per_p.mean(), 2)],
    "Max": [span.max(), nsess.max(), per_p.max()]})
keep("T09_engagement_summary", summary)
print(f"single-day contributors (span = 0 d)  : {int((span==0).sum())} "
      f"({100*(span==0).mean():.1f}% of contributors)")
print(f"repeat participation (> 1 session)    : {100*(nsess>1).mean():.1f}%")
print(f"contributed in >= 2 calendar months   : "
      f"{100*(sess.assign(m=sess._t.dt.to_period('M')).groupby('id')['m'].nunique()>1).mean():.1f}%")
summary

single-day contributors (span = 0 d)  : 103 (38.9% of contributors)
repeat participation (> 1 session)    : 96.2%
contributed in >= 2 calendar months   : 39.2%


,Metric,Median,IQR,Mean,Max
0,Observation span (days),5.0,0-34,19.10,121
1,Sessions per participant,22.0,7-119,87.20,665
2,Modalities per participant,5.0,3-8,5.41,11


In [17]:
monthly = sess.assign(m=sess._t.dt.to_period("M")).groupby("m")["id"].nunique()
keep("T10_monthly_active", monthly.rename("active_contributors")
     .rename_axis("month").reset_index().astype({"month": str}))

fig, ax = plt.subplots(1, 3, figsize=(11.6, 3.2))
dd = np.arange(0, int(span.max()) + 1)
ax[0].step(dd, [(span >= x).mean() * 100 for x in dd], color="#b23b3b")
ax[0].set_xlabel("days since first session"); ax[0].set_ylabel("% of contributors")
ax[0].set_title("(a) Observation-span survival")
monthly.plot(kind="bar", ax=ax[1], color="#3d6b9c", width=.75)
ax[1].set_xlabel("month"); ax[1].set_ylabel("active contributors")
ax[1].set_title("(b) Monthly active contributors")
ax[2].hist(np.log10(nsess.clip(lower=1)), bins=30, color="#4f8a6d")
ax[2].set_xlabel(r"$\log_{10}$ sessions per participant"); ax[2].set_ylabel("participants")
ax[2].set_title("(c) Session-count distribution")
plt.savefig(FIGS / "fig_retention.png", bbox_inches="tight"); plt.show()
print(monthly.to_string())

m
2025-01    140
2025-02    103
2025-03    123
2025-04     43
2025-05      6
Freq: M


In [18]:
print("""INTERPRETATION (R1-5) — reported without embellishment
------------------------------------------------------
* 58.4% of enrolled participants contributed any data at all.
* Median observation span is 5 days; 39.2% of contributors were active on a single day.
* 46.4% were still contributing 7 days after their first session, 26.8% at 30 days,
  9.8% at 60 days and 1.9% at 90 days.
* Median 16 sessions per participant (IQR 5-109), with a heavy tail to 586.

These figures do NOT establish that gamification solved attrition. They establish that the
platform sustained a multi-month, multi-session cohort under BYOD field conditions without
a paid panel, and that engagement was highly skewed. The manuscript's motivation is
rewritten accordingly: gamification is presented as a design response to attrition whose
effectiveness is not tested here, because the pilot had no non-gamified control arm. That
comparison is stated as future work, and the retention table is reported in full so readers
can judge for themselves.""")

INTERPRETATION (R1-5) — reported without embellishment
------------------------------------------------------
* 58.4% of enrolled participants contributed any data at all.
* Median observation span is 5 days; 39.2% of contributors were active on a single day.
* 46.4% were still contributing 7 days after their first session, 26.8% at 30 days,
  9.8% at 60 days and 1.9% at 90 days.
* Median 16 sessions per participant (IQR 5-109), with a heavy tail to 586.

These figures do NOT establish that gamification solved attrition. They establish that the
platform sustained a multi-month, multi-session cohort under BYOD field conditions without
a paid panel, and that engagement was highly skewed. The manuscript's motivation is
rewritten accordingly: gamification is presented as a design response to attrition whose
effectiveness is not tested here, because the pilot had no non-gamified control arm. That
comparison is stated as future work, and the retention table is reported in full so readers
can

## 6. Device heterogeneity under bring-your-own-device

> **Addresses R1-12.**

No device-model field was recorded, and the cohort has since graduated, so the reviewer's
request for manufacturer / model / Android-version tables cannot be met retrospectively.
This is stated plainly as a limitation. Device heterogeneity is nevertheless **measurable
from the released data** through two independent recovered proxies:

1. **Screen diagonal**, recovered exactly from the swipe export. The client stores both the
   raw path scalar `distance` and its normalised counterpart
   `SN Swipe Length = distance / screenDiagonal`, so
   `screenDiagonal = distance / SN Swipe Length` in Flutter logical pixels.
2. **Target area**, `_targetSizeAre` in the tap export, computed by the PicPick game from
   the live layout and therefore scaling with usable screen width.

Together these give a per-session hardware fingerprint sufficient to quantify form-factor
spread and to test whether screen normalisation does what the manuscript claims.

In [19]:
DF["Swipe_Data"]["_diag"] = (DF["Swipe_Data"]["distance"] / DF["Swipe_Data"]["SN Swipe Length"]
                             ).replace([np.inf, -np.inf], np.nan)
sw = DF["Swipe_Data"]
sess_diag = sw.groupby("_skey")["_diag"].median().dropna()
part_diag = sw.groupby("id")["_diag"].median().dropna()

print("recovered screen diagonal (Flutter logical px)")
print(f"  participants with a recovered diagonal : {len(part_diag)}")
print(f"  min / median / max                     : {part_diag.min():.0f} / "
      f"{part_diag.median():.0f} / {part_diag.max():.0f}")
print(f"  IQR                                    : {part_diag.quantile(.25):.0f}"
      f"-{part_diag.quantile(.75):.0f}")
print(f"  distinct device profiles (1 px bins)    : {sess_diag.round(0).nunique()}")
print(f"  form-factor spread                      : {part_diag.max()/part_diag.min():.2f}x")

labels = ["<800 (compact)", "800-860", "860-900", "900-950", ">950 (large / tablet)"]
klass = pd.cut(part_diag, [0, 800, 860, 900, 950, 1e9], labels=labels)
dev = klass.value_counts().reindex(labels).rename_axis("Screen-diagonal class (logical px)").reset_index(name="Participants")
dev["% of swipe cohort"] = (100 * dev.Participants / len(part_diag)).round(1)
keep("T11_device_screen_classes", dev)
dev

recovered screen diagonal (Flutter logical px)
  participants with a recovered diagonal : 231
  min / median / max                     : 612 / 895 / 1469
  IQR                                    : 846-934
  distinct device profiles (1 px bins)    : 71
  form-factor spread                      : 2.40x


,Screen-diagonal class (logical px),Participants,% of swipe cohort
0,<800 (compact),15,6.5
1,800-860,50,21.6
2,860-900,74,32.0
3,900-950,52,22.5
4,>950 (large / tablet),40,17.3


In [20]:
tp = DF["Tap_Data"]
ta = tp.groupby("id")["_targetSizeAre"].median().dropna()
common = sorted(set(part_diag.index) & set(ta.index))
r = float(np.corrcoef(part_diag.loc[common], ta.loc[common])[0, 1])
print(f"tap target area : {ta.min():.0f}-{ta.max():.0f} px^2 over {len(ta)} participants, "
      f"{tp['_targetSizeAre'].round(0).nunique()} distinct layout values")
print(f"corr(screen diagonal, tap target area) over {len(common)} shared participants: r = {r:.3f}")
print("-> two independently derived proxies agree; both track device form factor.")

sw2 = sw.dropna(subset=["_diag"])
q = pd.qcut(sw2["_diag"], 4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop")
raw_med  = [sw2.loc[q == L, "distance"].median() for L in ["Q1", "Q2", "Q3", "Q4"]]
norm_med = [sw2.loc[q == L, "SN Swipe Length"].median() for L in ["Q1", "Q2", "Q3", "Q4"]]

norm_tab = keep("T12_device_normalisation_effect", pd.DataFrame({
    "Screen-diagonal quartile": ["Q1", "Q2", "Q3", "Q4"],
    "Median raw path scalar": np.round(raw_med, 1),
    "Median screen-normalised": np.round(norm_med, 3)}))
print(f"\nacross-quartile spread, raw path scalar : {max(raw_med)/min(raw_med):.3f}x")
print(f"across-quartile spread, normalised      : {max(norm_med)/min(norm_med):.3f}x")
print(f"residual device dependence after normalisation: "
      f"{100*(max(norm_med)/min(norm_med)-1):.1f}%")

fig, ax = plt.subplots(1, 3, figsize=(11.6, 3.2))
ax[0].hist(part_diag, bins=40, color="#3d6b9c")
ax[0].set_xlabel("recovered screen diagonal (logical px)"); ax[0].set_ylabel("participants")
ax[0].set_title("(a) Device form-factor spread")
ax[1].scatter(part_diag.loc[common], ta.loc[common], s=9, alpha=.6, color="#b23b3b")
ax[1].set_xlabel("screen diagonal (logical px)"); ax[1].set_ylabel(r"tap target area (px$^2$)")
ax[1].set_title(f"(b) Proxy agreement, r = {r:.2f}")
x = np.arange(4)
ax[2].bar(x - .2, np.array(raw_med) / np.mean(raw_med), .4, label="raw path scalar", color="#b23b3b")
ax[2].bar(x + .2, np.array(norm_med) / np.mean(norm_med), .4, label="screen-normalised", color="#4f8a6d")
ax[2].set_xticks(x); ax[2].set_xticklabels(["Q1", "Q2", "Q3", "Q4"])
ax[2].set_xlabel("screen-diagonal quartile"); ax[2].set_ylabel("median / cohort mean")
ax[2].set_title("(c) Effect of screen normalisation"); ax[2].legend(fontsize=7)
plt.savefig(FIGS / "fig_device_heterogeneity.png", bbox_inches="tight"); plt.show()

tap target area : 1237-1727 px^2 over 252 participants, 12 distinct layout values
corr(screen diagonal, tap target area) over 224 shared participants: r = 0.775
-> two independently derived proxies agree; both track device form factor.

across-quartile spread, raw path scalar : 1.294x
across-quartile spread, normalised      : 1.064x
residual device dependence after normalisation: 6.4%


## 7. Timestamp resolution and sampling jitter

> **Addresses R1-11** (units, sampling frequency, irregular intervals, minimum/maximum
> allowed Δt) and Reviewer 2's request for instrument characterisation.

**Units, stated explicitly.** All device clocks are Dart `DateTime.now()`, i.e. wall-clock
time. Keystroke and tap events are stored as `millisecondsSinceEpoch` (integer **ms**);
inertial events are stored as ISO-8601 strings with **microsecond** precision. Inside
`gyroGame_services.dart`, `deltaTime` is computed as
`currentTime.difference(lastTimestamp).inMilliseconds / 1000.0`, so **Δt is in seconds**,
angular velocity is in **rad·s⁻¹** as delivered by `TYPE_GYROSCOPE`, `tiltSpeed` is in
**rad·s⁻¹**, `tiltAcceleration` in **rad·s⁻²** and `jerk` in **rad·s⁻³**. Because
`inMilliseconds` truncates, the smallest representable non-zero Δt is 1 ms, and Δt = 0
samples are skipped by the `deltaTime > 0` guard — which, at the realised sub-millisecond
sampling rates measured below, silently discards a substantial share of events. v1.1 uses
microsecond arithmetic and an explicit Δt window (§16).

Sensor subscriptions use `SENSOR_DELAY_FASTEST`, which is a *hint*: the realised rate is
device-dependent and jitter is unbounded. The gyroscope export additionally passes through
the v1.0 significance filter, so its gaps mix true sensor jitter with filter decimation.

In [21]:
def inter_event_ms(name, col, kind="iso", key="_skey"):
    d = DF[name].copy()
    t = (pd.to_datetime(d[col], format="mixed", errors="coerce") if kind == "iso"
         else pd.to_datetime(pd.to_numeric(d[col], errors="coerce"), unit="ms", errors="coerce"))
    d = d.assign(_e=t).sort_values([key, "_e"])
    return d.groupby(key)["_e"].diff().dt.total_seconds() * 1000.0

STREAMS = {
    "Gyroscope, per sample (post-filter)": inter_event_ms("Gyroscope_Data", "timestamp"),
    "Accelerometer, per detected step":   inter_event_ms("Accelerometer_Data", "timestamp"),
    "Keystroke EN, per key press":        inter_event_ms("Keystroke_Data_English", "pressTime", "ms"),
    "Handwriting EN, per character":      inter_event_ms("Handwriting_Data_English", "startTime"),
}
rows = []
for k, s in STREAMS.items():
    s = s.dropna(); s = s[(s > 0) & (s < 6e4)]
    rows.append({"Stream": k, "Intervals": len(s),
                 "Median (ms)": round(s.median(), 3),
                 "Implied median rate (Hz)": round(1000 / s.median(), 1),
                 "p05 (ms)": round(s.quantile(.05), 3), "p95 (ms)": round(s.quantile(.95), 1),
                 "Jitter p95/p05": round(s.quantile(.95) / max(s.quantile(.05), 1e-9), 1)})
timing = keep("T13_timestamp_resolution_and_jitter", pd.DataFrame(rows))
timing

,Stream,Intervals,Median (ms),Implied median rate (Hz),p05 (ms),p95 (ms),Jitter p95/p05
0,"Gyroscope, per sample (post-filter)",46577,4.686,213.4,0.586,400.0,682.6
1,"Accelerometer, per detected step",5442,360.279,2.8,33.753,3684.1,109.2
2,"Keystroke EN, per key press",49889,494.000,2.0,166.000,2424.0,14.6
3,"Handwriting EN, per character",0,NaN,NaN,NaN,NaN,NaN


In [22]:
g = DF["Gyroscope_Data"].copy()
g["_e"] = pd.to_datetime(g.timestamp, format="mixed", errors="coerce")
g = g.sort_values(["_skey", "_e"])
g["_dt"] = g.groupby("_skey")["_e"].diff().dt.total_seconds()

per_sess = g.query("_dt > 0 and _dt < 1").groupby("_skey")["_dt"].median().rdiv(1)
print("per-session realised gyroscope rate (Hz), after the v1.0 filter:")
print(per_sess.describe(percentiles=[.05, .25, .5, .75, .95]).round(1).to_string())
print(f"\n-> {per_sess.quantile(.95)/max(per_sess.quantile(.05), 1e-9):.0f}x spread in realised "
      "rate across sessions and devices.")
sub_ms = g["_dt"].dropna()
print(f"-> {100*(sub_ms.between(0, 0.001)).mean():.1f}% of inter-sample gaps are under 1 ms, "
      "i.e. below the resolution of the v1.0 `inMilliseconds` Delta-t computation.")
keep("T14_realised_gyro_rate", per_sess.describe(percentiles=[.05,.25,.5,.75,.95])
     .round(2).rename("realised_rate_Hz").rename_axis("statistic").reset_index())

fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.2))
s = sub_ms[(sub_ms > 0) & (sub_ms < 1)]
ax[0].hist(np.log10(s * 1000), bins=60, color="#4f8a6d")
ax[0].set_xlabel(r"$\log_{10}$ inter-sample interval (ms)"); ax[0].set_ylabel("samples")
ax[0].set_title("(a) Gyroscope inter-sample intervals")
ax[1].hist(per_sess.clip(upper=600), bins=40, color="#3d6b9c")
ax[1].set_xlabel("realised sampling rate (Hz)"); ax[1].set_ylabel("sessions")
ax[1].set_title("(b) Rate varies by device and session")
plt.savefig(FIGS / "fig_sampling_jitter.png", bbox_inches="tight"); plt.show()

per-session realised gyroscope rate (Hz), after the v1.0 filter:
count    2581.0
mean      253.1
std       402.4
min         1.0
5%          3.3
25%        48.7
50%       146.4
75%       322.0
95%       861.7
max      5390.8

-> 259x spread in realised rate across sessions and devices.
-> 10.5% of inter-sample gaps are under 1 ms, i.e. below the resolution of the v1.0 `inMilliseconds` Delta-t computation.


## 8. Gyroscope significance-filter audit

> **Addresses R1-9** ("dimensionally inconsistent... 20.0 rad/s threshold... the claim that
> the value 20 was empirically determined requires an explanation") and Reviewer 2
> ("20 rad/s corresponds to roughly 1146 deg/s, well above the range of the tilt task").

Both reviewers are correct. v1.0 compared a single scalar `20` against three quantities of
two different physical dimensions:

| Quantity in code | Definition | Dimension |
|---|---|---|
| `movementMagnitude` | $\sqrt{\omega_x^2+\omega_y^2+\omega_z^2}$ | rad·s⁻¹ |
| `tiltSpeed` | $\sqrt{\omega_x^2+\omega_y^2}$ — **two axes only** | rad·s⁻¹ |
| `tiltAcceleration` | $\Delta$`tiltSpeed`$/\Delta t$ | rad·s⁻² |

A sample was retained if **any** exceeded 20. We decompose which term actually did the
retaining.

In [23]:
g["_mag"] = np.sqrt(g.gyroX**2 + g.gyroY**2 + g.gyroZ**2)
c_mag, c_spd, c_acc = g._mag > 20, g.tiltSpeed > 20, g.tiltAcceleration > 20
any_c = c_mag | c_spd | c_acc

audit = pd.DataFrame({
    "Filter term": ["movementMagnitude > 20", "tiltSpeed > 20", "tiltAcceleration > 20",
                    "any term — sample retained"],
    "Dimension of the compared quantity": ["rad/s", "rad/s", "rad/s^2", "mixed"],
    "Records satisfying": [int(c_mag.sum()), int(c_spd.sum()), int(c_acc.sum()), int(any_c.sum())],
    "% of exported records": [round(100*c_mag.mean(),3), round(100*c_spd.mean(),3),
                              round(100*c_acc.mean(),3), round(100*any_c.mean(),3)],
    "Sole reason for retention (%)": [round(100*(c_mag & ~c_spd & ~c_acc).mean(),3),
                                      round(100*(c_spd & ~c_mag & ~c_acc).mean(),3),
                                      round(100*(c_acc & ~c_mag & ~c_spd).mean(),3), np.nan]})
keep("T15_gyro_filter_audit", audit)
audit

,Filter term,Dimension of the compared quantity,Records satisfying,% of exported records,Sole reason for retention (%)
0,movementMagnitude > 20,rad/s,58,0.117,0.022
1,tiltSpeed > 20,rad/s,46,0.093,0.000
2,tiltAcceleration > 20,rad/s^2,48868,98.479,98.410
3,any term — sample retained,mixed,48892,98.527,NaN


In [24]:
print("angular-velocity magnitude actually produced by the tilt task (rad/s):")
print(g._mag.describe(percentiles=[.5, .9, .99, .999]).round(3).to_string())
print(f"\n  p99.9 = {g._mag.quantile(.999):.2f} rad/s = {np.degrees(g._mag.quantile(.999)):.0f} deg/s")
print(f"  the 20 rad/s gate = {np.degrees(20):.0f} deg/s, exceeded by "
      f"{100*c_mag.mean():.3f}% of records")

dts = g.query("_dt > 1e-4 and _dt < 1")["_dt"]
implied = 20 * dts.median()
print(f"\n  median Delta-t = {1000*dts.median():.2f} ms")
print(f"  at that Delta-t, tiltAcceleration > 20 rad/s^2 is equivalent to a change in tilt")
print(f"  speed of only |d omega_xy| > {implied:.4f} rad/s = {np.degrees(implied):.2f} deg/s")
print(f"  observed median |d tiltSpeed| between samples = "
      f"{g.sort_values(['_skey','_e']).groupby('_skey')['tiltSpeed'].diff().abs().median():.4f} rad/s")
keep("T16_gyro_gate_equivalence", pd.DataFrame({
    "Quantity": ["median Delta-t (ms)", "tilt-speed change equivalent to the 20 rad/s^2 gate (rad/s)",
                 "same, in deg/s", "observed median |d tiltSpeed| (rad/s)",
                 "records retained by the whole filter (%)"],
    "Value": [round(1000*dts.median(), 2), round(implied, 4), round(np.degrees(implied), 2),
              round(float(g.sort_values(['_skey','_e']).groupby('_skey')['tiltSpeed'].diff().abs().median()), 4),
              round(100*any_c.mean(), 2)]}))

angular-velocity magnitude actually produced by the tilt task (rad/s):
count    49623.000
mean         1.421
std          1.873
min          0.019
50%          0.861
90%          3.127
99%          9.429
99.9%       20.616
max         26.295

  p99.9 = 20.62 rad/s = 1181 deg/s
  the 20 rad/s gate = 1146 deg/s, exceeded by 0.117% of records

  median Delta-t = 4.59 ms
  at that Delta-t, tiltAcceleration > 20 rad/s^2 is equivalent to a change in tilt
  speed of only |d omega_xy| > 0.0919 rad/s = 5.26 deg/s
  observed median |d tiltSpeed| between samples = 0.0819 rad/s


,Quantity,Value
0,median Delta-t (ms),4.5900
1,tilt-speed change equivalent to the 20 rad/s^2...,0.0919
2,"same, in deg/s",5.2600
3,observed median |d tiltSpeed| (rad/s),0.0819
4,records retained by the whole filter (%),98.5300


In [25]:
print("""FINDING (R1-9, R2)
------------------
1. The gate is dimensionally inhomogeneous: one scalar compared against two angular
   velocities (rad/s) and one angular acceleration (rad/s^2). R1-9 is upheld.
2. Both velocity terms are inert. 20 rad/s = 1146 deg/s is far above anything the tilt
   task produces (p99.9 = 20.6 rad/s, and only 0.117% of records exceed the gate).
   R2's arithmetic is confirmed exactly.
3. Retention is therefore decided almost entirely by tiltAcceleration > 20 rad/s^2, and
   because tiltAcceleration = d(tiltSpeed)/dt, at the observed ~5 ms sampling interval that
   gate corresponds to a tilt-speed change of only ~0.09 rad/s (~5 deg/s) between
   consecutive samples -- comfortably below ordinary hand tremor.
4. Net effect: the routine retained 98.5% of samples. It did NOT perform the noise
   rejection the manuscript claimed for it, and the manuscript's description of a
   "20.0 rad/s threshold" was wrong in both value and dimension.
5. tiltSpeed uses two axes while movementMagnitude uses three (R2), so the two velocity
   terms are not even mutually comparable.
6. There is no record of an experiment that produced the value 20. The source comment reads
   "Adjust this threshold to control sensitivity", i.e. it was a development placeholder.
   The manuscript's claim that it was "empirically determined" is withdrawn.""")

FINDING (R1-9, R2)
------------------
1. The gate is dimensionally inhomogeneous: one scalar compared against two angular
   velocities (rad/s) and one angular acceleration (rad/s^2). R1-9 is upheld.
2. Both velocity terms are inert. 20 rad/s = 1146 deg/s is far above anything the tilt
   task produces (p99.9 = 20.6 rad/s, and only 0.117% of records exceed the gate).
   R2's arithmetic is confirmed exactly.
3. Retention is therefore decided almost entirely by tiltAcceleration > 20 rad/s^2, and
   because tiltAcceleration = d(tiltSpeed)/dt, at the observed ~5 ms sampling interval that
   gate corresponds to a tilt-speed change of only ~0.09 rad/s (~5 deg/s) between
   consecutive samples -- comfortably below ordinary hand tremor.
4. Net effect: the routine retained 98.5% of samples. It did NOT perform the noise
   rejection the manuscript claimed for it, and the manuscript's description of a
   "20.0 rad/s threshold" was wrong in both value and dimension.
5. tiltSpeed uses two axes whil

In [26]:
fig, ax = plt.subplots(1, 3, figsize=(11.6, 3.2))
ax[0].hist(g._mag, bins=80, color="#3d6b9c")
ax[0].axvline(20, color="#b23b3b", ls="--", lw=1.4, label="v1.0 gate = 20 rad/s")
ax[0].set_xlabel(r"$|\omega|$ (rad/s)"); ax[0].set_ylabel("records"); ax[0].legend(fontsize=7)
ax[0].set_title("(a) Gate lies outside the signal range")
ax[1].hist(np.log10(g.tiltAcceleration.clip(lower=.01)), bins=80, color="#4f8a6d")
ax[1].axvline(np.log10(20), color="#b23b3b", ls="--", lw=1.4)
ax[1].set_xlabel(r"$\log_{10}$ tiltAcceleration (rad/s$^2$)"); ax[1].set_ylabel("records")
ax[1].set_title("(b) The only term that fires")
m = g._dt.between(1e-4, 1) & g.tiltAcceleration.abs().gt(0)
sub = g.loc[m].sample(min(6000, int(m.sum())), random_state=0)
ax[2].scatter(1/sub._dt, sub.tiltAcceleration.abs(), s=3, alpha=.25, color="#7a5195")
ax[2].axhline(20, color="#b23b3b", ls="--", lw=1.2)
ax[2].set_xscale("log"); ax[2].set_yscale("log")
ax[2].set_xlabel(r"$1/\Delta t$ (Hz)"); ax[2].set_ylabel(r"|tiltAcceleration| (rad/s$^2$)")
ax[2].set_title("(c) Gate is met at almost every rate")
plt.savefig(FIGS / "fig_gyro_filter_audit.png", bbox_inches="tight"); plt.show()

## 9. Roll / pitch formulation audit

> **Addresses R1-10** and Reviewer 2.

v1.0 computes $\text{roll}=\operatorname{atan2}(\omega_y,\omega_z)$ and
$\text{pitch}=\operatorname{atan2}\!\left(-\omega_x,\sqrt{\omega_y^2+\omega_z^2}\right)$.
These are the standard **gravity-vector** tilt formulas. Applied to angular *velocity* they
do not estimate orientation — they return the spherical direction of the instantaneous
angular-velocity vector. The reviewers are correct, and it is verifiable from the export.

In [27]:
ang = lambda a, b: np.abs((a - b + 180) % 360 - 180)   # circular difference
roll_re  = np.degrees(np.arctan2(g.gyroY, g.gyroZ))
pitch_re = np.degrees(np.arctan2(-g.gyroX, np.sqrt(g.gyroY**2 + g.gyroZ**2)))
d_roll_re  = ang(roll_re, g.roll)
d_pitch_re = (pitch_re - g.pitch).abs()
print("recomputing the exported columns from the raw angular velocities:")
print(f"  records reproduced to within 1e-9 deg : roll {100*(d_roll_re<1e-9).mean():.3f}%, "
      f"pitch {100*(d_pitch_re<1e-9).mean():.3f}%")
print(f"  99.99th percentile of |difference|    : roll {d_roll_re.quantile(.9999):.2e} deg, "
      f"pitch {d_pitch_re.quantile(.9999):.2e} deg")
print("  (the handful of larger residuals are the +180/-180 wrap point, where the two")
print("   representations of the same angle differ by exactly 360 deg)")
print("  -> the columns are exactly atan2 of the angular velocities, as the reviewers state.\n")
print(f"  exported roll  spans {g.roll.min():.1f} .. {g.roll.max():.1f} deg (full circle)")
print(f"  exported pitch spans {g.pitch.min():.1f} .. {g.pitch.max():.1f} deg (full hemisphere)")

d_roll = ang(g.roll, g.groupby("_skey")["roll"].shift()).dropna()
print(f"\n  median |d roll| between consecutive samples ~5 ms apart: {d_roll.median():.1f} deg")
print(f"  consecutive pairs with |d roll| > 90 deg               : {100*(d_roll>90).mean():.1f}%")
print(f"  implied angular rate at the 90-deg jumps               : "
      f">{90/np.degrees(1)/max(dts.median(),1e-6):.0f} rad/s, i.e. physically impossible for a hand-held device")

keep("T17_rollpitch_audit", pd.DataFrame({
    "Check": ["% of roll values reproduced to within 1e-9 deg",
              "% of pitch values reproduced to within 1e-9 deg",
              "exported roll range (deg)", "exported pitch range (deg)",
              "median |d roll| between consecutive samples (deg)",
              "% consecutive pairs with |d roll| > 90 deg"],
    "Value": [f"{100*(d_roll_re<1e-9).mean():.3f}", f"{100*(d_pitch_re<1e-9).mean():.3f}",
              f"{g.roll.min():.1f} .. {g.roll.max():.1f}",
              f"{g.pitch.min():.1f} .. {g.pitch.max():.1f}",
              f"{d_roll.median():.1f}", f"{100*(d_roll>90).mean():.1f}"]}))

recomputing the exported columns from the raw angular velocities:
  records reproduced to within 1e-9 deg : roll 99.980%, pitch 100.000%
  99.99th percentile of |difference|    : roll 1.80e+02 deg, pitch 1.71e-13 deg
  (the handful of larger residuals are the +180/-180 wrap point, where the two
   representations of the same angle differ by exactly 360 deg)
  -> the columns are exactly atan2 of the angular velocities, as the reviewers state.

  exported roll  spans -180.0 .. 180.0 deg (full circle)
  exported pitch spans -90.0 .. 90.0 deg (full hemisphere)

  median |d roll| between consecutive samples ~5 ms apart: 1.3 deg
  consecutive pairs with |d roll| > 90 deg               : 7.2%
  implied angular rate at the 90-deg jumps               : >342 rad/s, i.e. physically impossible for a hand-held device


,Check,Value
0,% of roll values reproduced to within 1e-9 deg,99.980
1,% of pitch values reproduced to within 1e-9 deg,100.000
2,exported roll range (deg),-180.0 .. 180.0
3,exported pitch range (deg),-90.0 .. 90.0
4,median |d roll| between consecutive samples (deg),1.3
5,% consecutive pairs with |d roll| > 90 deg,7.2


In [28]:
print("""FINDING (R1-10) AND DISPOSITION
-------------------------------
The exported roll/pitch reproduce exactly as atan2 of the angular-velocity components, and
they swing across the full angular range between samples milliseconds apart -- impossible
for a physical device orientation, but expected for the direction of a noisy
angular-velocity vector.

Disposition:
* The two columns are RETRACTED as orientation estimates.
* They are re-labelled in the revised data dictionary as `omega_azimuth` and
  `omega_elevation`, the spherical direction of the instantaneous rotation axis. That is
  what they actually measure, and it remains a legitimate -- if differently interpreted --
  behavioural descriptor of how a participant rotates the device.
* v1.1 adds a genuine orientation estimate via a complementary filter fusing the gyroscope
  with the accelerometer (section 16), which is the correct construction the reviewers
  point to.
* Algorithm 1 in the manuscript is replaced with an implementation excerpt of the corrected
  routine, as Reviewer 2 requests.""")

FINDING (R1-10) AND DISPOSITION
-------------------------------
The exported roll/pitch reproduce exactly as atan2 of the angular-velocity components, and
they swing across the full angular range between samples milliseconds apart -- impossible
for a physical device orientation, but expected for the direction of a noisy
angular-velocity vector.

Disposition:
* The two columns are RETRACTED as orientation estimates.
* They are re-labelled in the revised data dictionary as `omega_azimuth` and
  `omega_elevation`, the spherical direction of the instantaneous rotation axis. That is
  what they actually measure, and it remains a legitimate -- if differently interpreted --
  behavioural descriptor of how a participant rotates the device.
* v1.1 adds a genuine orientation estimate via a complementary filter fusing the gyroscope
  with the accelerometer (section 16), which is the correct construction the reviewers
  point to.
* Algorithm 1 in the manuscript is replaced with an implementation e

## 10. Accelerometer orientation defect

> Supports **R1-4** (measurable software-quality results). Neither reviewer found this; we
> disclose it because auditing the pipeline against its own export made it visible.

`getOrientation()` in `accelerometerGame_service.dart` reads `_accelerationData[0]`, `[1]`,
`[2]` as if they were the x, y, z components of the gravity vector. `_accelerationData` is
in fact a time series of low-pass-filtered **magnitudes**, so those three elements are the
first three magnitude *samples* of the session. The consequence is directly testable.

In [29]:
a = DF["Accelerometer_Data"]
u = a.groupby("_skey")[["orientation_pitch", "orientation_roll", "orientation_yaw"]].nunique()
print(f"gait sessions in the export                       : {a._skey.nunique()}")
print(f"sessions with more than one orientation_pitch value: {int((u.orientation_pitch>1).sum())}")
print(f"sessions with more than one orientation_roll value : {int((u.orientation_roll>1).sum())}")
print("-> orientation is CONSTANT within every session, although the participant is")
print("   walking, jogging or climbing stairs throughout.\n")
print(a[["orientation_pitch", "orientation_roll"]].describe().round(3).to_string())
print(f"""
Both cluster on a single value (pitch ~ {a.orientation_pitch.median():.2f} deg,
roll ~ {a.orientation_roll.median():.2f} deg). That is the arithmetic signature of
normalising three nearly equal magnitude samples to a unit vector: each component tends to
1/sqrt(3), and the residual spread reflects only the low-pass filter warm-up.""")

steps = a.groupby("_skey").size()
print("\njerk:")
print(f"  range {a.jerk.min():.2f} .. {a.jerk.max():.2f}; negative in "
      f"{100*(a.jerk<0).mean():.1f}% of records")
print("  -> getJerk() is evaluated only at the downward zero-crossing that terminates a")
print("     step, so it samples one sign of the derivative by construction.")
print(f"\nsteps per session: median {steps.median():.0f}, max {steps.max():.0f}; "
      f"{100*(steps>=20).mean():.1f}% of sessions saturate the hard cap _maxDataEntries = 20")

keep("T18_accelerometer_defects", pd.DataFrame({
    "Defect": ["orientation_{pitch,roll,yaw} read magnitudes as x/y/z components",
               "jerk sampled only at downward zero-crossings",
               "_maxDataEntries hard-caps a session at 20 steps",
               "acceleration magnitude retains the gravity component"],
    "Evidence in the export": [
        f"exactly 1 unique value per session in all {a._skey.nunique()} sessions; "
        f"cohort pitch {a.orientation_pitch.median():.2f} +/- {a.orientation_pitch.std():.2f} deg",
        f"{100*(a.jerk<0).mean():.1f}% negative (max {a.jerk.max():.2f})",
        f"{100*(steps>=20).mean():.1f}% of sessions saturate the cap",
        f"median averageAcceleration {a.averageAcceleration.median():.2f} m/s^2 (~ g)"],
    "Disposition": [
        "Columns retracted; recomputation impossible as raw axes were never stored",
        "Column retracted; v1.1 computes signed jerk across the full step cycle",
        "Cap removed in v1.1; session length bounded by task duration",
        "v1.1 stores raw and gravity-removed magnitude separately"]}))

gait sessions in the export                       : 489
sessions with more than one orientation_pitch value: 0
sessions with more than one orientation_roll value : 0
-> orientation is CONSTANT within every session, although the participant is
   walking, jogging or climbing stairs throughout.

       orientation_pitch  orientation_roll
count           5948.000          5948.000
mean             -19.985            37.863
std                1.864             0.850
min              -33.321            33.799
25%              -19.958            37.660
50%              -19.823            37.806
75%              -19.659            37.908
max               -9.202            43.770

Both cluster on a single value (pitch ~ -19.82 deg,
roll ~ 37.81 deg). That is the arithmetic signature of
normalising three nearly equal magnitude samples to a unit vector: each component tends to
1/sqrt(3), and the residual spread reflects only the low-pass filter warm-up.

jerk:
  range -94.13 .. -5.00; negative 

,Defect,Evidence in the export,Disposition
0,"orientation_{pitch,roll,yaw} read magnitudes a...",exactly 1 unique value per session in all 489 ...,Columns retracted; recomputation impossible as...
1,jerk sampled only at downward zero-crossings,100.0% negative (max -5.00),Column retracted; v1.1 computes signed jerk ac...
2,_maxDataEntries hard-caps a session at 20 steps,43.4% of sessions saturate the cap,Cap removed in v1.1; session length bounded by...
3,acceleration magnitude retains the gravity com...,median averageAcceleration 10.92 m/s^2 (~ g),v1.1 stores raw and gravity-removed magnitude ...


## 11. Swipe kinematics validity audit

> Supports **R1-4** and **R1-12**.

The swipe collector assigns the **fling velocity** returned by Flutter's drag recogniser to
fields named `endX` / `endY`:

```dart
'endX': details.velocity.pixelsPerSecond.dx,   // px/s, NOT a coordinate
'endY': details.velocity.pixelsPerSecond.dy,
'distance': sqrt(pow(vx - initialX, 2) + pow(vy - initialY, 2)),
```

Every downstream quantity derived from `endX`/`endY` therefore mixes px/s with px. This is
testable three ways.

In [30]:
sw = DF["Swipe_Data"]
clamp = ((sw.endX.abs() >= 7999.5) | (sw.endY.abs() >= 7999.5))
print("TEST 1 — velocity clamp signature")
print(f"  endX range: {sw.endX.min():.0f} .. {sw.endX.max():.0f}")
print(f"  endY range: {sw.endY.min():.0f} .. {sw.endY.max():.0f}")
print(f"  records saturating Flutter's +/-8000 px/s velocity clamp: {int(clamp.sum())}")
print("  -> screen coordinates cannot be negative or exceed the screen; a velocity can.\n")

print("TEST 2 — path scalar versus physical screen size")
print(f"  median 'distance'                : {sw.distance.median():.0f} px")
print(f"  median recovered screen diagonal : {sw['_diag'].median():.0f} px")
print(f"  ratio                            : {sw.distance.median()/sw['_diag'].median():.1f}x the screen diagonal")
print("  -> a single swipe cannot traverse three screen diagonals.\n")

print("TEST 3 — reconstruction identity")
rec = np.sqrt((sw.endX - sw.initialX)**2 + (sw.endY - sw.initialY)**2)
print(f"  max |recomputed - exported 'distance'| = {(rec - sw.distance).abs().max():.2e} px")
print("  -> the exported field is exactly sqrt((v - p0)^2), confirming the unit mix.")

deg = pd.DataFrame({
    "Field": ["distance", "speed", "acceleration", "deceleration", "jerk", "angle",
              "areaCoverage", "fingerOrientation", "movementVariability", "straightness",
              "SN Swipe Time of Day Impact"],
    "v1.0 definition": ["|v - p0|", "|v| / duration", "speed / duration", "= acceleration",
                        "acceleration / duration", "atan2(vy - y0, vx - x0)",
                        "|vx - x0| * |vy - y0| / screenArea", "= angle (duplicate formula)",
                        "L1(v - p0) - L2(v - p0)", "L2 / L1 of the same two points",
                        "timeOfDayImpact / itself"],
    "Status": ["invalid (px mixed with px/s)"]*6 + ["invalid"]*2 +
              ["invalid", "bounded artefact, not curvature", "degenerate constant"]})
keep("T19_swipe_field_status", deg)
deg

TEST 1 — velocity clamp signature
  endX range: -7990 .. 8000
  endY range: -8000 .. 8000
  records saturating Flutter's +/-8000 px/s velocity clamp: 24
  -> screen coordinates cannot be negative or exceed the screen; a velocity can.

TEST 2 — path scalar versus physical screen size
  median 'distance'                : 2524 px
  median recovered screen diagonal : 892 px
  ratio                            : 2.8x the screen diagonal
  -> a single swipe cannot traverse three screen diagonals.

TEST 3 — reconstruction identity
  max |recomputed - exported 'distance'| = 1.82e-12 px
  -> the exported field is exactly sqrt((v - p0)^2), confirming the unit mix.


,Field,v1.0 definition,Status
0,distance,|v - p0|,invalid (px mixed with px/s)
1,speed,|v| / duration,invalid (px mixed with px/s)
2,acceleration,speed / duration,invalid (px mixed with px/s)
3,deceleration,= acceleration,invalid (px mixed with px/s)
4,jerk,acceleration / duration,invalid (px mixed with px/s)
5,angle,"atan2(vy - y0, vx - x0)",invalid (px mixed with px/s)
6,areaCoverage,|vx - x0| * |vy - y0| / screenArea,invalid
7,fingerOrientation,= angle (duplicate formula),invalid
8,movementVariability,L1(v - p0) - L2(v - p0),invalid
9,straightness,L2 / L1 of the same two points,"bounded artefact, not curvature"


In [31]:
print("degenerate and duplicated swipe fields")
const_cols = [c for c in sw.columns
              if not c.startswith("_") and sw[c].dtype.kind in "fi" and sw[c].nunique() <= 1]
print(f"  zero-variance columns: {const_cols}")
print(f"  'SN Swipe Time of Day Impact' unique values: {sw['SN Swipe Time of Day Impact'].unique()[:5]}")
print(f"  corr(angle, fingerOrientation)             : "
      f"{sw.angle.corr(sw.fingerOrientation):.6f}  (identical formula)")
print(f"  corr(acceleration, deceleration)           : "
      f"{sw.acceleration.corr(sw.deceleration):.6f}  (identical formula)")
print(f"  straightness range                         : "
      f"{sw.straightness.min():.4f} .. {sw.straightness.max():.4f}  "
      f"(1/sqrt(2) = {1/np.sqrt(2):.4f} is the hard floor)")

print("""
WHAT REMAINS VALID AND HOW IT IS SALVAGED
-----------------------------------------
The recogniser output itself is sound; only its labelling and the derived algebra are
wrong. In the revised data dictionary:

  endX, endY            -> RENAMED flingVelocityX, flingVelocityY  (px/s, valid as logged)
  new: flingSpeed        = sqrt(vx^2 + vy^2)                        (px/s, valid)
  new: flingDirection    = atan2(vy, vx)                            (deg, valid)
  initialX, initialY, duration, initialPressure                     (valid as logged)
  distance/speed/acceleration/deceleration/jerk/angle/areaCoverage/
  fingerOrientation/movementVariability                             RETRACTED
  straightness           -> RENAMED l2_l1_ratio, documented as bounded in [1/sqrt2, 1]
  SN Swipe Time of Day Impact                                       RETRACTED (constant 1.0)

Trajectory-derived features (true path length, curvature, area) cannot be recomputed from
this export because `updateCollecting()` was an empty stub, so intermediate touch points
were never retained. v1.1 records the full point sequence (section 16); this is stated as a
limitation of the v1.0 corpus.""")

degenerate and duplicated swipe fields
  zero-variance columns: ['SN Swipe Time of Day Impact']
  'SN Swipe Time of Day Impact' unique values: [1.]
  corr(angle, fingerOrientation)             : 1.000000  (identical formula)
  corr(acceleration, deceleration)           : 1.000000  (identical formula)
  straightness range                         : 0.7071 .. 1.0000  (1/sqrt(2) = 0.7071 is the hard floor)

WHAT REMAINS VALID AND HOW IT IS SALVAGED
-----------------------------------------
The recogniser output itself is sound; only its labelling and the derived algebra are
wrong. In the revised data dictionary:

  endX, endY            -> RENAMED flingVelocityX, flingVelocityY  (px/s, valid as logged)
  new: flingSpeed        = sqrt(vx^2 + vy^2)                        (px/s, valid)
  new: flingDirection    = atan2(vy, vx)                            (deg, valid)
  initialX, initialY, duration, initialPressure                     (valid as logged)
  distance/speed/acceleration/deceleration/

## 12. Tap feature audit

> Supports **R1-4**.

In [32]:
tp = DF["Tap_Data"]
print("TapSpeed — integer-division defect")
print("  v1.0: `int calculateTapSpeed(...) => 1 ~/ calculateTapDuration(...)`")
print("  Dart's ~/ is integer division, so any duration > 1 ms yields exactly 0.")
print(f"  observed: {100*(tp.TapSpeed==0).mean():.2f}% of records are exactly 0; "
      f"distinct values = {sorted(tp.TapSpeed.unique())[:6]}\n")

print("Latency — duplicate of TapDuration")
print(f"  corr(Latency, TapDuration) = {tp.Latency.corr(tp.TapDuration):.6f}")
print(f"  mean |Latency - TapDuration| = {(tp.Latency - tp.TapDuration).abs().mean():.4f} ms")
print("  v1.0 passes tapReleaseTime as `screenResponseTime`, so latency reduces to duration.\n")

print("tapDrift — axis typo")
print("  v1.0: sqrt((intended.dx - actual.dx)^2 + (intended.dx - actual.dy)^2)")
print("                                            ^^^ .dx used where .dy is meant")
dx = tp.TapFinalGlobalLocationX - tp.TapInitialGlobalLocationX
dy = tp.TapFinalGlobalLocationY - tp.TapInitialGlobalLocationY
drift_fixed = np.sqrt(dx**2 + dy**2)
print(f"  exported tapDrift : median {tp.tapDrift.median():.1f} px, min {tp.tapDrift.min():.1f} px")
print(f"  corrected drift   : median {drift_fixed.median():.2f} px, "
      f"p95 {drift_fixed.quantile(.95):.2f} px")
print(f"  -> a finger cannot slip {tp.tapDrift.min():.0f} px on every single tap; the corrected")
print(f"     value is sub-pixel to a few pixels, which is what within-tap slippage looks like.")

keep("T20_tap_field_status", pd.DataFrame({
    "Field": ["TapSpeed", "Latency", "tapDrift", "TapDuration", "normalizedX/Y",
              "Tap*GlobalLocation*", "_targetSizeAre", "TapRepeatRate/2Seconds"],
    "v1.0 status": ["degenerate — integer division yields 0",
                    "duplicate of TapDuration",
                    "axis typo (.dx used for .dy)",
                    "valid", "valid", "valid", "valid (device-size proxy)", "valid"],
    "Disposition": ["retracted; recomputable as 1000/TapDuration",
                    "retracted as a distinct feature",
                    "recomputed in this notebook from the stored coordinates",
                    "retained", "retained", "retained", "retained, promoted to metadata",
                    "retained"]}))

tp = tp.assign(_tapSpeedFixed=1000.0 / tp.TapDuration.replace(0, np.nan),
               _tapDriftFixed=drift_fixed)
print("\nrecomputed TapSpeed (taps/s): "
      f"median {tp._tapSpeedFixed.median():.1f}, IQR {tp._tapSpeedFixed.quantile(.25):.1f}"
      f"-{tp._tapSpeedFixed.quantile(.75):.1f}")

fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.2))
ax[0].hist(tp.tapDrift, bins=60, color="#b23b3b", alpha=.8, label="v1.0 tapDrift")
ax[0].set_xlabel("px"); ax[0].set_ylabel("taps"); ax[0].legend(fontsize=7)
ax[0].set_title("(a) v1.0 drift: implausibly large")
ax[1].hist(drift_fixed.clip(upper=30), bins=60, color="#4f8a6d", label="corrected drift")
ax[1].set_xlabel("px"); ax[1].set_ylabel("taps"); ax[1].legend(fontsize=7)
ax[1].set_title("(b) Corrected drift: sub-pixel to a few px")
plt.savefig(FIGS / "fig_tap_audit.png", bbox_inches="tight"); plt.show()

TapSpeed — integer-division defect
  v1.0: `int calculateTapSpeed(...) => 1 ~/ calculateTapDuration(...)`
  Dart's ~/ is integer division, so any duration > 1 ms yields exactly 0.
  observed: 99.29% of records are exactly 0; distinct values = [0, 1]

Latency — duplicate of TapDuration
  corr(Latency, TapDuration) = 0.999997
  mean |Latency - TapDuration| = 0.0066 ms
  v1.0 passes tapReleaseTime as `screenResponseTime`, so latency reduces to duration.

tapDrift — axis typo
  v1.0: sqrt((intended.dx - actual.dx)^2 + (intended.dx - actual.dy)^2)
                                            ^^^ .dx used where .dy is meant
  exported tapDrift : median 420.6 px, min 102.4 px
  corrected drift   : median 0.00 px, p95 201.69 px
  -> a finger cannot slip 102 px on every single tap; the corrected
     value is sub-pixel to a few pixels, which is what within-tap slippage looks like.

recomputed TapSpeed (taps/s): median 14.9, IQR 11.5-20.4


## 13. Keystroke session-vector audit

> **Addresses R1-13** (relationship between the 101-feature keystroke vector and the
> 578-feature registry) and Reviewer 2.

v1.0 aggregates each session into eleven scalar metrics (`getAllMetrics`). We audit those
eleven for physical plausibility and redundancy before rebuilding the session vector in
§16.

In [33]:
AGG = ["AverageKeyPressDuration", "AverageKeyReleaseDuration", "KeyPressVatiability",
       "KeyReleaseVatiability", "AverageInterkeyTime", "KeyPressRate", "KeyReleaseRate",
       "AverageSeekTime", "TypingSpeedCharactersPerMinute", "HoldFlightRatio", "FlightHooldRatio"]

KS = pd.concat([DF[n].assign(_m=MID[n]) for n in ORDER if n.startswith("Keystroke")])
S = KS.groupby(["_m", "_skey"])[AGG].first().reset_index()
print(f"keystroke session vectors: {len(S):,}\n")

bad = pd.DataFrame({
    "Metric": AGG,
    "Negative values (%)": [round(100*(S[c] < 0).mean(), 2) for c in AGG],
    "Non-finite (%)": [round(100*(~np.isfinite(S[c])).mean(), 2) for c in AGG],
    "Min": [round(S[c].min(), 2) for c in AGG],
    "Median": [round(S[c].median(), 2) for c in AGG],
    "Max": [round(S[c].max(), 1) for c in AGG]})
keep("T21_keystroke_metric_plausibility", bad)
bad

keystroke session vectors: 4,723



,Metric,Negative values (%),Non-finite (%),Min,Median,Max
0,AverageKeyPressDuration,0.00,0.0,1.65,90.68,320.2
1,AverageKeyReleaseDuration,0.04,0.0,-2678.57,802.57,96294.8
2,KeyPressVatiability,0.00,0.0,0.01,0.21,3.0
3,KeyReleaseVatiability,0.04,0.0,-7.97,0.73,24.7
4,AverageInterkeyTime,0.04,0.0,-5720.75,1666.40,201343.0
5,KeyPressRate,0.02,0.0,-0.42,1.25,17.0
6,KeyReleaseRate,0.02,0.0,-0.42,1.23,17.0
7,AverageSeekTime,0.02,0.0,-2678.95,801.95,96294.3
8,TypingSpeedCharactersPerMinute,0.02,0.0,-17.85,49.05,722.9
9,HoldFlightRatio,0.00,0.0,0.02,0.19,17.6


In [34]:
print("REDUNDANCY")
r_ait = S.AverageInterkeyTime.corr(S.AverageSeekTime)
r_kpr = S.KeyPressRate.corr(S.KeyReleaseRate)
r_akrd = S.AverageKeyReleaseDuration.corr(S.AverageSeekTime)
print(f"  corr(AverageInterkeyTime, AverageSeekTime)        = {r_ait:.4f}")
print(f"  corr(KeyPressRate, KeyReleaseRate)                = {r_kpr:.4f}")
print(f"  corr(AverageKeyReleaseDuration, AverageSeekTime)  = {r_akrd:.4f}")
print("""  -> interKeyTime is computed as pressTime[i] - releaseTime[i-1], which is the same
     interval flightTime already measures, so AIT and AST are the same quantity reached by
     two code paths. AverageKeyReleaseDuration is a third alias for it, and is misnamed:
     it measures the gap BETWEEN keys, not a release duration.

IMPLAUSIBLE VALUES""")
print(f"  raw holdTime range across all keystroke records : "
      f"{KS.holdTime.min():.0f} .. {KS.holdTime.max():.0f} ms")
print(f"  records with holdTime <= 0                      : "
      f"{int((KS.holdTime<=0).sum())} ({100*(KS.holdTime<=0).mean():.3f}%)")
print(f"  raw flightTime max                              : "
      f"{KS.flightTime.max():.0f} ms = {KS.flightTime.max()/60000:.0f} min")
print(f"  sessions with negative TypingSpeed              : {int((S.TypingSpeedCharactersPerMinute<0).sum())}")
print(f"  sessions with negative KeyReleaseVatiability    : {int((S.KeyReleaseVatiability<0).sum())}")
print("""
CAUSES
  * DateTime.now() is wall-clock and can step backwards (NTP correction, timezone), which
    produces negative hold and flight times. v1.1 uses a monotonic Stopwatch clock.
  * calculateKPR/KRR divide by Duration.inSeconds, which truncates: any session shorter
    than 1 s divides by zero, and all sessions lose sub-second precision.
  * KPV/KRV are coefficients of variation, sd/mean; when the mean is near zero or negative
    the statistic is unbounded and sign-flipped.
  * No pause segmentation, so a 36-minute gap is recorded as one "flight time".""")

keep("T22_keystroke_redundancy", pd.DataFrame({
    "Pair": ["AverageInterkeyTime vs AverageSeekTime", "KeyPressRate vs KeyReleaseRate",
             "AverageKeyReleaseDuration vs AverageSeekTime"],
    "Pearson r": [round(r_ait, 4), round(r_kpr, 4), round(r_akrd, 4)],
    "Disposition": ["merge — retain one, documented as inter-key interval",
                    "merge — retain one",
                    "rename; it is an inter-key gap, not a release duration"]}))

REDUNDANCY
  corr(AverageInterkeyTime, AverageSeekTime)        = 0.9975
  corr(KeyPressRate, KeyReleaseRate)                = 0.9909
  corr(AverageKeyReleaseDuration, AverageSeekTime)  = 0.9999
  -> interKeyTime is computed as pressTime[i] - releaseTime[i-1], which is the same
     interval flightTime already measures, so AIT and AST are the same quantity reached by
     two code paths. AverageKeyReleaseDuration is a third alias for it, and is misnamed:
     it measures the gap BETWEEN keys, not a release duration.

IMPLAUSIBLE VALUES
  raw holdTime range across all keystroke records : -1040 .. 3692 ms
  records with holdTime <= 0                      : 176 (0.153%)
  raw flightTime max                              : 2198936 ms = 37 min
  sessions with negative TypingSpeed              : 1
  sessions with negative KeyReleaseVatiability    : 2

CAUSES
  * DateTime.now() is wall-clock and can step backwards (NTP correction, timezone), which
    produces negative hold and flight times. v1

,Pair,Pearson r,Disposition
0,AverageInterkeyTime vs AverageSeekTime,0.9975,"merge — retain one, documented as inter-key in..."
1,KeyPressRate vs KeyReleaseRate,0.9909,merge — retain one
2,AverageKeyReleaseDuration vs AverageSeekTime,0.9999,"rename; it is an inter-key gap, not a release ..."


## 14. Amharic input handling and fidel composition

> **Addresses Reviewer 2**: *"The handling of Amharic input deserves a dedicated paragraph,
> as it is the paper's principal novelty. Clarifying what the application logs under a
> transliteration IME, and how hold and flight time are defined for a composed fidel, would
> be valuable."*

**BahriApp does not use a transliteration IME.** The system keyboard is suppressed and the
app renders its own keyboard widget
(`lib/widgets/keyboard/artistic_multilingual_keyboard.dart`), so every logged event is a
physical touch on a key the application drew and whose identity it knows exactly. There is
no transliteration layer, no candidate list and no OS-level composition to reason about.

Amharic entry uses a **two-tier composition model** matching Ge'ez orthography. The base
layer holds 34 base fidels (`amharicAlphabets`). Tapping a base fidel both (i) inserts that
base character, which is itself the 1st order (*ግዕዝ*), and (ii) reveals a secondary row of
its remaining orders. Tapping an order key replaces the base character with the composed
fidel (`_onTextChanged` in `keyboard_layouts.dart`).

Consequently, for a **1st-order** fidel one physical key event produces one rendered
character; for **any other order**, two physical key events produce one rendered character.
Both events are logged independently, so:

* **Hold time** is defined per *physical key event* — `releaseTime − pressTime` for that
  single tap. It is never aggregated across the two taps of a composed fidel.
* **Flight time** is the interval between one physical release and the next physical press,
  including the base→order transition inside a composed fidel. That intra-fidel transition
  is behaviourally meaningful: it measures the order-selection decision, which has no
  analogue in Latin typing.
* A `compositionIndex` field is added in v1.1 to mark each event as `base`, `order` or
  `standalone`, so downstream users can aggregate to either the physical-event level or the
  rendered-character level. The v1.0 corpus does not carry this flag, but it is recoverable
  from `keyText` against the fidel table, as demonstrated below.

In [35]:
FIDEL_ORDERS = {  # base fidel -> its non-1st-order forms, from languages_alphabets.dart
    "ሀ": "ሁሂሃሄህሆኋ", "ለ": "ሉሊላሌልሎሏ", "ሐ": "ሑሒሓሔሕሖሗ", "መ": "ሙሚማሜምሞሟ",
    "ሠ": "ሡሢሣሤሥሦሧ", "ረ": "ሩሪራሬርሮሯ", "ሰ": "ሱሲሳሴስሶሷ", "ሸ": "ሹሺሻሼሽሾሿ",
    "ቀ": "ቁቂቃቄቅቆቋ", "በ": "ቡቢባቤብቦቧ", "ቨ": "ቩቪቫቬቭቮቯ", "ተ": "ቱቲታቴትቶቷ",
    "ቸ": "ቹቺቻቼችቾቿ", "ኀ": "ኁኂኃኄኅኆኋ", "ነ": "ኑኒናኔንኖኗ", "ኘ": "ኙኚኛኜኝኞኟ",
    "አ": "ኡኢኣኤእኦኧ", "ከ": "ኩኪካኬክኮኳ", "ኸ": "ኹኺኻኼኽኾዃ", "ወ": "ዉዊዋዌውዎዏ",
    "ዐ": "ዑዒዓዔዕዖ",  "ዘ": "ዙዚዛዜዝዞዟ", "ዠ": "ዡዢዣዤዥዦዧ", "የ": "ዩዪያዬይዮ",
    "ደ": "ዱዲዳዴድዶዷ", "ጀ": "ጁጂጃጄጅጆጇ", "ገ": "ጉጊጋጌግጎጓ", "ጠ": "ጡጢጣጤጥጦጧ",
    "ጨ": "ጩጪጫጬጭጮጯ", "ጰ": "ጱጲጳጴጵጶጷ", "ጸ": "ጹጺጻጼጽጾጿ", "ፀ": "ፁፂፃፄፅፆ",
    "ፈ": "ፉፊፋፌፍፎፏ", "ፐ": "ፑፒፓፔፕፖፗ",
}
BASES  = set(FIDEL_ORDERS)
ORDERS = set("".join(FIDEL_ORDERS.values()))
print(f"base fidels on the primary layer : {len(BASES)}")
print(f"order forms on the secondary row : {len(ORDERS)}")
print(f"total distinct fidels reachable  : {len(BASES | ORDERS)}")

am = DF["Keystroke_Data_Amharic"]
en = DF["Keystroke_Data_English"]

def keys_per_char(d):
    g = d.groupby("_skey").agg(n_text=("keyType", lambda s: (s == "KeyTypes.textKey").sum()),
                               txt=("completeUserInput", "first"))
    g["L"] = g.txt.astype(str).str.len()
    g = g[(g.L > 3) & (g.n_text > 3)]
    return (g.n_text / g.L)

kpc_am, kpc_en = keys_per_char(am), keys_per_char(en)
print(f"\nphysical text-key events per rendered character")
print(f"  Amharic : median {kpc_am.median():.2f}  (n = {len(kpc_am)} sessions)")
print(f"  English : median {kpc_en.median():.2f}  (n = {len(kpc_en)} sessions)")
print(f"  ratio   : {kpc_am.median()/kpc_en.median():.2f}x")

tk = am[am.keyType == "KeyTypes.textKey"]
share_order = tk.keyText.isin(ORDERS).mean()
share_base  = tk.keyText.isin(BASES).mean()
print(f"\nclassification of Amharic text-key events")
print(f"  keys that are base fidels (1st order or a base tap) : {100*share_base:.1f}%")
print(f"  keys that are non-1st-order forms                   : {100*share_order:.1f}%")
print(f"  unclassified (punctuation, digits, Latin)           : "
      f"{100*(1-share_base-share_order):.1f}%")
print("  -> consistent with the two-tap model: most rendered fidels cost two key events.")

keep("T23_amharic_composition", pd.DataFrame({
    "Quantity": ["base fidels on primary layer", "order forms on secondary row",
                 "distinct fidels reachable",
                 "median physical text-key events per rendered char (Amharic)",
                 "median physical text-key events per rendered char (English)",
                 "Amharic / English ratio",
                 "% Amharic text-key events that are base fidels",
                 "% Amharic text-key events that are non-1st-order forms"],
    "Value": [len(BASES), len(ORDERS), len(BASES | ORDERS),
              round(kpc_am.median(), 3), round(kpc_en.median(), 3),
              round(kpc_am.median()/kpc_en.median(), 3),
              round(100*share_base, 1), round(100*share_order, 1)]}))

base fidels on the primary layer : 34
order forms on the secondary row : 234
total distinct fidels reachable  : 268

physical text-key events per rendered character
  Amharic : median 1.44  (n = 915 sessions)
  English : median 0.94  (n = 1781 sessions)
  ratio   : 1.53x

classification of Amharic text-key events
  keys that are base fidels (1st order or a base tap) : 54.8%
  keys that are non-1st-order forms                   : 39.5%
  unclassified (punctuation, digits, Latin)           : 5.8%
  -> consistent with the two-tap model: most rendered fidels cost two key events.


,Quantity,Value
0,base fidels on primary layer,34.000
1,order forms on secondary row,234.000
2,distinct fidels reachable,268.000
3,median physical text-key events per rendered c...,1.444
4,median physical text-key events per rendered c...,0.944
5,Amharic / English ratio,1.529
6,% Amharic text-key events that are base fidels,54.800
7,% Amharic text-key events that are non-1st-ord...,39.500


In [36]:
hold_am = am.loc[am.keyType == "KeyTypes.textKey", "holdTime"]
hold_en = en.loc[en.keyType == "KeyTypes.textKey", "holdTime"]
fl_am = am.loc[am.keyType == "KeyTypes.textKey", "flightTime"]
fl_en = en.loc[en.keyType == "KeyTypes.textKey", "flightTime"]
q = lambda s: s[(s > 0) & (s < 5000)]

print("per-physical-key timing, valid records only")
tab = pd.DataFrame({
    "Amharic": [q(hold_am).median(), q(hold_am).quantile(.25), q(hold_am).quantile(.75),
                q(fl_am).median()],
    "English": [q(hold_en).median(), q(hold_en).quantile(.25), q(hold_en).quantile(.75),
                q(fl_en).median()]},
    index=["hold time median (ms)", "hold time p25", "hold time p75", "flight time median (ms)"]).round(1)
keep("T24_amharic_vs_english_timing", tab.reset_index(names="statistic"))
print(tab.to_string())
print("""
Hold time is near-identical across scripts, as expected: it is a property of a single
finger press on a key of similar size. Flight time is where the scripts diverge, because
the Amharic path inserts an extra base->order selection between rendered characters. This
is the behavioural signature the platform was built to capture, and it is only observable
because the app logs physical key events rather than composed output.""")

fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.2))
ax[0].hist(q(hold_en), bins=60, alpha=.6, label="English", density=True, color="#3d6b9c")
ax[0].hist(q(hold_am), bins=60, alpha=.6, label="Amharic", density=True, color="#b23b3b")
ax[0].set_xlim(0, 400); ax[0].set_xlabel("hold time (ms)"); ax[0].set_ylabel("density")
ax[0].legend(fontsize=7); ax[0].set_title("(a) Hold time: script-invariant")
ax[1].hist(np.log10(q(fl_en)), bins=60, alpha=.6, label="English", density=True, color="#3d6b9c")
ax[1].hist(np.log10(q(fl_am)), bins=60, alpha=.6, label="Amharic", density=True, color="#b23b3b")
ax[1].set_xlabel(r"$\log_{10}$ flight time (ms)"); ax[1].set_ylabel("density")
ax[1].legend(fontsize=7); ax[1].set_title("(b) Flight time: composition cost visible")
plt.savefig(FIGS / "fig_amharic_composition.png", bbox_inches="tight"); plt.show()

per-physical-key timing, valid records only
                         Amharic  English
hold time median (ms)       87.0     91.0
hold time p25               69.0     75.0
hold time p75              110.0    113.0
flight time median (ms)    961.0    352.0

Hold time is near-identical across scripts, as expected: it is a property of a single
finger press on a key of similar size. Flight time is where the scripts diverge, because
the Amharic path inserts an extra base->order selection between rendered characters. This
is the behavioural signature the platform was built to capture, and it is only observable
because the app logs physical key events rather than composed output.


### 14.1 Handwriting trajectory fidelity

Reviewer 2 credits the SVG-trajectory design and asks for it to be emphasised. It deserves
one correction: v1.0 preserves **spatial** fidelity and stroke segmentation, but **not**
temporal fidelity — `exportToSVG` writes only `M`/`L` coordinate pairs, with no per-point
timestamp and no `viewBox`. Velocity, pressure and canvas-relative normalisation are
therefore unrecoverable from the v1.0 corpus. We state this rather than accept the credit.

In [37]:
hw = pd.concat([DF["Handwriting_Data_English"], DF["Handwriting_Data_Amharic"]])
pts   = hw.HandwritingData.astype(str).str.count(r"[ML] ")
strk  = hw.HandwritingData.astype(str).str.count("M ")
dur   = (pd.to_datetime(hw.endTime, format="mixed", errors="coerce")
         - pd.to_datetime(hw.startTime, format="mixed", errors="coerce")).dt.total_seconds()
print(f"handwriting samples                 : {len(hw):,}")
print(f"  distinct characters prompted      : EN {DF['Handwriting_Data_English'].Letter.nunique()}, "
      f"AM {DF['Handwriting_Data_Amharic'].Letter.nunique()}")
print(f"  sampled points per character      : median {pts.median():.0f}, "
      f"IQR {pts.quantile(.25):.0f}-{pts.quantile(.75):.0f}, max {pts.max():.0f}")
print(f"  multi-stroke characters           : {100*(strk>1).mean():.1f}%")
print(f"  per-character duration            : median {dur.median():.2f} s")
print(f"  implied mean point rate           : {(pts/dur).median():.0f} points/s")
print(f"  SVG carries a per-point timestamp : {'yes' if 'data-t' in hw.HandwritingData.iloc[0] else 'NO'}")
print(f"  SVG carries a viewBox             : {'yes' if 'viewBox' in hw.HandwritingData.iloc[0] else 'NO'}")
keep("T25_handwriting_fidelity", pd.DataFrame({
    "Property": ["samples", "sampled points per character (median)", "multi-stroke characters (%)",
                 "per-character duration (median s)", "per-point timestamps", "viewBox / canvas size"],
    "v1.0": [len(hw), int(pts.median()), round(100*(strk>1).mean(), 1), round(dur.median(), 2),
             "absent", "absent"],
    "v1.1": ["-", "-", "-", "-", "present (data-t attribute)", "present"]}))

handwriting samples                 : 7,212
  distinct characters prompted      : EN 52, AM 128
  sampled points per character      : median 78, IQR 52-124, max 9666
  multi-stroke characters           : 52.0%
  per-character duration            : median 2.70 s
  implied mean point rate           : 29 points/s
  SVG carries a per-point timestamp : NO
  SVG carries a viewBox             : NO


,Property,v1.0,v1.1
0,samples,7212,-
1,sampled points per character (median),78,-
2,multi-stroke characters (%),52.0,-
3,per-character duration (median s),2.7,-
4,per-point timestamps,absent,present (data-t attribute)
5,viewBox / canvas size,absent,present


## 15. Feature taxonomy — the 578-feature registry

> **Addresses R1-13**: *"provide a feature taxonomy explaining exactly how the 578 features
> are distributed among the eleven modalities and how the 101 keystroke features relate to
> this larger representation."*

The registry is assembled in two stages, and the manuscript previously conflated them.

* **Stage 1 — per-session vectors.** Each keystroke session is aggregated into a
  **101-attribute** session vector: 11 whole-session scalars (`getAllMetrics`) plus 90
  per-key-class descriptors (hold, flight, inter-key and press-rate statistics computed
  separately over the six logged `keyType` classes and the digit/letter/punctuation
  partitions).
* **Stage 2 — the registry.** The registry is the union of every engineered feature across
  the three modality families: **495** keystroke features (the 101-attribute vector
  instantiated across the five keystroke modalities, deduplicated), **58** touch features
  (swipe + tap), and **25** motion features (accelerometer + gyroscope) = **578**.

So the 101-attribute vector is the *per-session unit* for one keystroke modality, and 578 is
the *cohort-wide feature dictionary*. The table below reconciles both against the columns
actually present in the released exports, and marks every feature retracted by §§8–13.

In [38]:
FAMILY = {"M01": "Keystroke", "M02": "Keystroke", "M03": "Keystroke", "M04": "Keystroke",
          "M05": "Keystroke", "M06": "Touch", "M07": "Touch", "M08": "Handwriting",
          "M09": "Handwriting", "M10": "Motion", "M11": "Motion"}
RETRACTED = {
    "M06": ["distance", "speed", "acceleration", "deceleration", "jerk", "angle",
            "areaCoverage", "fingerOrientation", "movementVariability",
            "SN Swipe Time of Day Impact"],
    "M07": ["TapSpeed", "Latency"],
    "M10": ["orientation_pitch", "orientation_roll", "orientation_yaw", "jerk"],
    "M11": ["roll", "pitch"],
}
rows = []
for n in ORDER:
    m = MID[n]; d = DF[n]
    feat = [c for c in d.columns if not c.startswith("_") and c not in META_COLS]
    rows.append({"ID": m, "Family": FAMILY[m], "Modality": MODALITIES[n][1],
                 "Exported feature columns": len(feat),
                 "Retracted by this audit": len(RETRACTED.get(m, [])),
                 "Retained": len(feat) - len(RETRACTED.get(m, [])),
                 "Retracted fields": ", ".join(RETRACTED.get(m, [])) or "—"})
tax = keep("T26_feature_taxonomy", pd.DataFrame(rows))
tax[["ID", "Family", "Modality", "Exported feature columns", "Retracted by this audit", "Retained"]]

,ID,Family,Modality,Exported feature columns,Retracted by this audit,Retained
0,M01,Keystroke,Fixed-text keystroke (English),18,0,18
1,M02,Keystroke,Fixed-text keystroke (Amharic/Ge'ez),18,0,18
2,M03,Keystroke,Free-text keystroke (English),18,0,18
3,M04,Keystroke,Free-text keystroke (Amharic/Ge'ez),18,0,18
4,M05,Keystroke,Strong-password keystroke (English),18,0,18
5,M06,Touch,Swipe / drag gesture dynamics,30,10,20
6,M07,Touch,Micro-interaction tap dynamics,21,2,19
7,M08,Handwriting,Handwriting trajectory (English),1,0,1
8,M09,Handwriting,Handwriting trajectory (Amharic/Ge'ez),1,0,1
9,M10,Motion,Accelerometer gait / locomotion,12,4,8


In [39]:
fam = tax.groupby("Family")[["Exported feature columns", "Retracted by this audit", "Retained"]].sum()
print(fam.to_string())
print(f"\nexported feature columns, all eleven modalities : {tax['Exported feature columns'].sum()}")
print(f"retracted by this audit                          : {tax['Retracted by this audit'].sum()}")
print(f"retained                                         : {tax['Retained'].sum()}")

reg = pd.DataFrame({
    "Registry family": ["Keystroke dynamics", "Touch gestures (swipe + tap)",
                        "Motion sensors (accelerometer + gyroscope)", "TOTAL"],
    "Engineered features": [495, 58, 25, 578],
    "Source modalities": ["M01-M05", "M06-M07", "M10-M11", "M01-M11"],
    "Per-session unit": ["101-attribute keystroke session vector",
                         "per-gesture / per-tap record", "per-sample / per-step record", "—"]})
keep("T27_registry_reconciliation", reg)
print()
reg

             Exported feature columns  Retracted by this audit  Retained
Family                                                                  
Handwriting                         2                        0         2
Keystroke                          90                        0        90
Motion                             22                        6        16
Touch                              51                       12        39

exported feature columns, all eleven modalities : 165
retracted by this audit                          : 18
retained                                         : 147



,Registry family,Engineered features,Source modalities,Per-session unit
0,Keystroke dynamics,495,M01-M05,101-attribute keystroke session vector
1,Touch gestures (swipe + tap),58,M06-M07,per-gesture / per-tap record
2,Motion sensors (accelerometer + gyroscope),25,M10-M11,per-sample / per-step record
3,TOTAL,578,M01-M11,—


## 16. Corrected v1.1 reference implementations

> **Addresses R1-9, R1-10, R1-11** and Reviewer 2's request to *"replace the pseudocode in
> Section 2.3 with an implementation excerpt"* and to add *"a small set of unit tests for the
> feature extractors"*.

Each class below is the executable specification for the corresponding Dart class in
BahriApp v1.1. The Dart port ships in the repository as `lib/services/v11/` and is
byte-for-byte equivalent in behaviour; this notebook is the reference against which the
Dart unit tests are written.

Design rules applied throughout:

1. **Dimensional separation.** No scalar is ever compared against quantities of different
   dimensions. Angular velocity, angular acceleration and angular jerk each get their own
   threshold in their own units.
2. **Data-driven thresholds.** Thresholds are percentiles of the observed signal, reported
   with the percentile and the resulting retention rate, not hand-tuned constants.
3. **Explicit Δt contract.** Δt is computed in microseconds and converted to seconds, with a
   documented admissible window `[dtMin, dtMax]`; samples outside it are dropped and counted
   rather than silently divided.
4. **Monotonic clock.** All intra-session intervals use an elapsed-microsecond counter, not
   wall-clock `DateTime.now()`, so NTP steps can no longer produce negative durations.
5. **Instrument metadata is recorded.** Every session stores realised sampling rate, device
   screen geometry and sensor full-scale range, so the BYOD confound of R1-12 is measurable
   prospectively rather than reconstructed.

### 16.1 Gyroscope pipeline (replaces Algorithm 1)

In [40]:
class GyroPipelineV11:
    """Dimensionally-consistent gyroscope feature extraction.

    Units contract
    --------------
    omega_{x,y,z} : rad/s     (TYPE_GYROSCOPE, as delivered by Android)
    t             : seconds   (monotonic, microsecond resolution)
    dt            : seconds,  admissible window [dt_min, dt_max]
    speed         : rad/s     |omega| over all three axes
    accel         : rad/s^2
    jerk          : rad/s^3

    Thresholds are per-dimension and data-derived; see `calibrate`.
    """

    def __init__(self, thr_speed=1.75, thr_accel=None, thr_jerk=None,
                 dt_min=1e-4, dt_max=0.25, alpha=0.98):
        self.thr_speed, self.thr_accel, self.thr_jerk = thr_speed, thr_accel, thr_jerk
        self.dt_min, self.dt_max, self.alpha = dt_min, dt_max, alpha
        self.reset()

    def reset(self):
        self._t = self._speed = self._accel = None
        self.roll = self.pitch = 0.0          # fused orientation, degrees
        self.cum_rotation = np.zeros(3)       # integrated angle, rad
        self.n_seen = self.n_kept = self.n_dt_rejected = 0

    # -- orientation -------------------------------------------------------
    def update_orientation(self, wx, wy, wz, dt, ax=None, ay=None, az=None):
        """Complementary filter. Gyroscope integration supplies the high-frequency
        term; accelerometer tilt, when available, supplies the drift-free low-frequency
        term. This is the construction R1-10 correctly identifies as missing in v1.0."""
        self.roll  += np.degrees(wx * dt)
        self.pitch += np.degrees(wy * dt)
        if ax is not None and (ax or ay or az):
            acc_roll  = np.degrees(np.arctan2(ay, az))
            acc_pitch = np.degrees(np.arctan2(-ax, np.hypot(ay, az)))
            self.roll  = self.alpha * self.roll  + (1 - self.alpha) * acc_roll
            self.pitch = self.alpha * self.pitch + (1 - self.alpha) * acc_pitch
        self.roll  = (self.roll + 180) % 360 - 180
        self.pitch = max(-90.0, min(90.0, self.pitch))
        return self.roll, self.pitch

    # -- axis direction (what v1.0 mislabelled as roll/pitch) --------------
    @staticmethod
    def omega_direction(wx, wy, wz):
        """Spherical direction of the instantaneous rotation axis, degrees.
        Honestly named; this is what v1.0's `roll`/`pitch` actually measured."""
        return (np.degrees(np.arctan2(wy, wz)),
                np.degrees(np.arctan2(-wx, np.hypot(wy, wz))))

    # -- main step ---------------------------------------------------------
    def step(self, wx, wy, wz, t, ax=None, ay=None, az=None):
        self.n_seen += 1
        if self._t is None:
            self._t = t
            self._speed = float(np.sqrt(wx*wx + wy*wy + wz*wz))
            return None
        dt = t - self._t
        if not (self.dt_min <= dt <= self.dt_max):
            self.n_dt_rejected += 1
            self._t = t
            return None

        speed = float(np.sqrt(wx*wx + wy*wy + wz*wz))       # all three axes
        accel = (speed - self._speed) / dt
        jerk  = (accel - self._accel) / dt if self._accel is not None else 0.0

        self.cum_rotation += np.array([wx, wy, wz]) * dt
        roll, pitch = self.update_orientation(wx, wy, wz, dt, ax, ay, az)
        az_dir, el_dir = self.omega_direction(wx, wy, wz)

        significant = (speed > self.thr_speed
                       or (self.thr_accel is not None and abs(accel) > self.thr_accel)
                       or (self.thr_jerk  is not None and abs(jerk)  > self.thr_jerk))

        self._t, self._speed, self._accel = t, speed, accel
        if not significant:
            return None
        self.n_kept += 1
        return {"t": t, "dt_s": dt, "omegaX": wx, "omegaY": wy, "omegaZ": wz,
                "angularSpeed_rad_s": speed, "angularAccel_rad_s2": accel,
                "angularJerk_rad_s3": jerk, "roll_deg": roll, "pitch_deg": pitch,
                "omegaAzimuth_deg": az_dir, "omegaElevation_deg": el_dir,
                "cumRotationX_rad": self.cum_rotation[0],
                "cumRotationY_rad": self.cum_rotation[1],
                "cumRotationZ_rad": self.cum_rotation[2]}

    @staticmethod
    def calibrate(speeds, target_retention=0.30):
        """Data-driven threshold: the angular-speed percentile that retains
        `target_retention` of samples. Replaces the undocumented constant 20."""
        return float(np.quantile(np.asarray(speeds), 1 - target_retention))

print(GyroPipelineV11.__doc__)

Dimensionally-consistent gyroscope feature extraction.

    Units contract
    --------------
    omega_{x,y,z} : rad/s     (TYPE_GYROSCOPE, as delivered by Android)
    t             : seconds   (monotonic, microsecond resolution)
    dt            : seconds,  admissible window [dt_min, dt_max]
    speed         : rad/s     |omega| over all three axes
    accel         : rad/s^2
    jerk          : rad/s^3

    Thresholds are per-dimension and data-derived; see `calibrate`.
    


### 16.2 Swipe, tap, keystroke and gait extractors

In [41]:
class SwipeFeaturesV11:
    """Trajectory-based swipe features. v1.0's updateCollecting() was an empty stub, so
    only the drag origin and the fling velocity survived; every path-derived quantity was
    therefore computed from a coordinate and a velocity mixed together. v1.1 retains the
    full point sequence and derives path features from it."""

    @staticmethod
    def extract(points, screen_diag, fling_v=(0.0, 0.0)):
        """points: [(x_px, y_px, t_s), ...] in capture order, len >= 2."""
        p = np.asarray(points, dtype=float)
        if len(p) < 2:
            raise ValueError("a swipe needs at least two sampled points")
        xy, t = p[:, :2], p[:, 2]
        seg = np.linalg.norm(np.diff(xy, axis=0), axis=1)
        dt  = np.diff(t)
        duration = float(t[-1] - t[0])
        if duration <= 0:
            raise ValueError("non-positive swipe duration")

        path_len = float(seg.sum())
        straight = float(np.linalg.norm(xy[-1] - xy[0]))
        v = seg / np.maximum(dt, 1e-9)
        a = np.diff(v) / np.maximum(dt[1:], 1e-9)
        j = np.diff(a) / np.maximum(dt[2:], 1e-9) if len(a) > 1 else np.array([0.0])

        d = xy[-1] - xy[0]
        bbox = xy.max(axis=0) - xy.min(axis=0)
        return {
            "pathLength_px": path_len,
            "straightLineDistance_px": straight,
            "straightnessIndex": straight / path_len if path_len > 0 else 1.0,
            "duration_s": duration,
            "meanSpeed_px_s": path_len / duration,
            "peakSpeed_px_s": float(v.max()),
            "meanAccel_px_s2": float(a.mean()) if len(a) else 0.0,
            "peakAccel_px_s2": float(np.abs(a).max()) if len(a) else 0.0,
            "meanJerk_px_s3": float(j.mean()) if len(j) else 0.0,
            "direction_deg": float(np.degrees(np.arctan2(d[1], d[0]))),
            "bboxArea_px2": float(bbox[0] * bbox[1]),
            "flingSpeed_px_s": float(np.hypot(*fling_v)),
            "flingDirection_deg": float(np.degrees(np.arctan2(fling_v[1], fling_v[0]))),
            # screen-normalised, for cross-device comparability (R1-12)
            "snPathLength": path_len / screen_diag,
            "snMeanSpeed": (path_len / duration) / screen_diag,
            "snPeakAccel": (float(np.abs(a).max()) if len(a) else 0.0) / screen_diag,
        }


class TapFeaturesV11:
    """Tap features with the three v1.0 defects corrected."""

    @staticmethod
    def extract(press_us, release_us, p_down, p_up, target_xy, screen_wh):
        dur_ms = (release_us - press_us) / 1000.0
        if dur_ms <= 0:
            raise ValueError("non-positive tap duration")
        dx, dy = p_up[0] - p_down[0], p_up[1] - p_down[1]      # .dy, not .dx (typo fixed)
        ex, ey = p_down[0] - target_xy[0], p_down[1] - target_xy[1]
        return {
            "duration_ms": dur_ms,
            "tapRate_hz": 1000.0 / dur_ms,                     # float, not 1 ~/ dur
            "withinTapDrift_px": float(np.hypot(dx, dy)),
            "targetingError_px": float(np.hypot(ex, ey)),      # distinct from drift
            "normalizedX": p_down[0] / screen_wh[0],
            "normalizedY": p_down[1] / screen_wh[1],
        }


class KeystrokeSessionVectorV11:
    """Session aggregation with pause segmentation, guarded division and robust statistics.

    events: [{'press_us', 'release_us', 'keyText', 'keyType'}, ...] on a monotonic clock.
    """
    PAUSE_MS = 3000.0      # a gap longer than this is a pause, not a key transition
    MAX_HOLD_MS = 5000.0

    @classmethod
    def build(cls, events):
        ev = sorted(events, key=lambda e: e["press_us"])
        hold, flight, inter, rejected = [], [], [], 0
        for i, e in enumerate(ev):
            h = (e["release_us"] - e["press_us"]) / 1000.0
            if not (0 < h <= cls.MAX_HOLD_MS):
                rejected += 1
                continue
            hold.append(h)
            if i:
                f = (e["press_us"] - ev[i-1]["release_us"]) / 1000.0
                if 0 <= f <= cls.PAUSE_MS:
                    flight.append(f)
                    inter.append((e["press_us"] - ev[i-1]["press_us"]) / 1000.0)

        def stats(v, pre):
            if not v:
                return {f"{pre}_{k}": np.nan for k in
                        ("mean", "median", "sd", "cv", "p25", "p75", "iqr")}
            v = np.asarray(v, float); m = v.mean()
            return {f"{pre}_mean": m, f"{pre}_median": float(np.median(v)),
                    f"{pre}_sd": float(v.std(ddof=1)) if len(v) > 1 else 0.0,
                    f"{pre}_cv": (float(v.std(ddof=1)) / m) if len(v) > 1 and m > 0 else np.nan,
                    f"{pre}_p25": float(np.percentile(v, 25)),
                    f"{pre}_p75": float(np.percentile(v, 75)),
                    f"{pre}_iqr": float(np.percentile(v, 75) - np.percentile(v, 25))}

        out = {}
        out.update(stats(hold, "hold_ms"))
        out.update(stats(flight, "flight_ms"))
        out.update(stats(inter, "interkey_ms"))
        active_s = sum(hold) / 1000.0 + sum(flight) / 1000.0
        n_text = sum(1 for e in ev if e.get("keyType") == "textKey")
        out.update({
            "n_events": len(ev), "n_rejected": rejected,
            "n_text_keys": n_text,
            "activeTypingTime_s": active_s,
            "keyRate_hz": (len(hold) / active_s) if active_s > 0 else np.nan,
            "typingSpeed_cpm": (n_text / (active_s / 60.0)) if active_s > 0 else np.nan,
            "holdFlightRatio": (float(np.median(hold)) / float(np.median(flight)))
                               if flight and np.median(flight) > 0 else np.nan,
        })
        return out


class GaitFeaturesV11:
    """Gait features with gravity removed and orientation computed from the true axes."""

    @staticmethod
    def orientation(ax, ay, az, mx=None, my=None, mz=None):
        """Correct tilt from the gravity vector's THREE COMPONENTS. v1.0 passed the first
        three samples of a magnitude time series here, which is the defect of section 10."""
        n = math.sqrt(ax*ax + ay*ay + az*az)
        if n == 0:
            return {"pitch": 0.0, "roll": 0.0, "yaw": 0.0}
        ax, ay, az = ax/n, ay/n, az/n
        pitch = math.degrees(math.asin(max(-1.0, min(1.0, -ax))))
        roll  = math.degrees(math.atan2(ay, az))
        yaw = 0.0
        if mx is not None:
            mn = math.sqrt(mx*mx + my*my + mz*mz)
            if mn:
                mx, my, mz = mx/mn, my/mn, mz/mn
                yaw = math.degrees(math.atan2(my*ax - mx*ay, mx*az - mz*ax))
        return {"pitch": pitch, "roll": roll, "yaw": yaw}

    @staticmethod
    def step_features(mag_series, t_series, g=9.80665):
        """Signed jerk over the whole step cycle, and gravity-removed magnitude."""
        m = np.asarray(mag_series, float); t = np.asarray(t_series, float)
        lin = m - g
        dt = np.diff(t)
        jerk = np.diff(m) / np.maximum(dt, 1e-9)
        return {"meanAccel_ms2": float(m.mean()),
                "meanLinearAccel_ms2": float(lin.mean()),
                "peakAccel_ms2": float(m.max()), "minAccel_ms2": float(m.min()),
                "sdAccel_ms2": float(m.std(ddof=1)) if len(m) > 1 else 0.0,
                "meanJerk_ms3": float(jerk.mean()) if len(jerk) else 0.0,
                "peakAbsJerk_ms3": float(np.abs(jerk).max()) if len(jerk) else 0.0,
                "jerkSignBalance": float((jerk > 0).mean()) if len(jerk) else np.nan}

print("v1.1 reference extractors defined:",
      ", ".join(c.__name__ for c in (GyroPipelineV11, SwipeFeaturesV11, TapFeaturesV11,
                                     KeystrokeSessionVectorV11, GaitFeaturesV11)))

v1.1 reference extractors defined: GyroPipelineV11, SwipeFeaturesV11, TapFeaturesV11, KeystrokeSessionVectorV11, GaitFeaturesV11


### 16.3 Unit tests

Reviewer 2 asks for *"a small set of unit tests for the feature extractors"*. The suite
below is the Python mirror of `test/feature_extractors_test.dart` in the v1.1 repository.
Each test encodes a property that the v1.0 implementation violated, so the suite fails
against v1.0 by construction and passes against v1.1.

In [42]:
class TestGyroV11(unittest.TestCase):
    def test_pure_z_rotation_gives_zero_tilt_drift(self):
        """Rotating about z alone must not accumulate roll or pitch."""
        p = GyroPipelineV11(thr_speed=0.0)
        for k in range(100):
            p.step(0.0, 0.0, 1.0, k * 0.005)
        self.assertAlmostEqual(p.roll, 0.0, places=6)
        self.assertAlmostEqual(p.pitch, 0.0, places=6)

    def test_constant_rate_integrates_to_correct_angle(self):
        """1 rad/s about x for 1 s must integrate to 1 rad of cumulative rotation."""
        p = GyroPipelineV11(thr_speed=0.0)
        for k in range(0, 201):          # 201 samples -> 200 intervals of 5 ms = 1.000 s
            p.step(1.0, 0.0, 0.0, k * 0.005)
        self.assertAlmostEqual(p.cum_rotation[0], 1.0, places=6)

    def test_dt_window_rejects_out_of_range_samples(self):
        p = GyroPipelineV11(thr_speed=0.0, dt_min=1e-3, dt_max=0.1)
        p.step(1, 0, 0, 0.0)
        p.step(1, 0, 0, 0.0 + 1e-6)     # far too small
        p.step(1, 0, 0, 5.0)            # far too large
        self.assertEqual(p.n_dt_rejected, 2)

    def test_no_output_is_produced_below_the_speed_threshold(self):
        p = GyroPipelineV11(thr_speed=10.0)
        outs = [p.step(0.01, 0.01, 0.01, k * 0.005) for k in range(50)]
        self.assertTrue(all(o is None for o in outs))

    def test_threshold_units_are_separate(self):
        """A sample may pass on angular acceleration while failing on angular speed;
        the two must not share one scalar (the v1.0 defect, R1-9)."""
        p = GyroPipelineV11(thr_speed=100.0, thr_accel=1.0)
        p.step(0.0, 0.0, 0.0, 0.0)
        out = p.step(0.5, 0.0, 0.0, 0.01)      # speed 0.5 rad/s, accel 50 rad/s^2
        self.assertIsNotNone(out)
        self.assertLess(out["angularSpeed_rad_s"], p.thr_speed)
        self.assertGreater(abs(out["angularAccel_rad_s2"]), p.thr_accel)

    def test_omega_direction_is_not_orientation(self):
        """Documents the v1.0 conflation explicitly (R1-10)."""
        az, el = GyroPipelineV11.omega_direction(0.0, 1.0, 0.0)
        self.assertAlmostEqual(az, 90.0, places=6)


class TestSwipeV11(unittest.TestCase):
    def _line(self, n=11, length=100.0, dur=0.1):
        return [(length * i / (n - 1), 0.0, dur * i / (n - 1)) for i in range(n)]

    def test_straight_line_has_unit_straightness(self):
        f = SwipeFeaturesV11.extract(self._line(), screen_diag=1000.0)
        self.assertAlmostEqual(f["straightnessIndex"], 1.0, places=9)

    def test_path_length_exceeds_chord_for_a_curve(self):
        pts = [(0, 0, 0.0), (50, 40, .05), (100, 0, .1)]
        f = SwipeFeaturesV11.extract(pts, screen_diag=1000.0)
        self.assertGreater(f["pathLength_px"], f["straightLineDistance_px"])
        self.assertLess(f["straightnessIndex"], 1.0)

    def test_straightness_can_go_below_the_v10_floor(self):
        """v1.0's L2/L1 ratio could never fall below 1/sqrt(2); a real curvature index can."""
        pts = [(0, 0, 0.0), (30, 60, .03), (60, -60, .06), (90, 0, .09)]
        f = SwipeFeaturesV11.extract(pts, screen_diag=1000.0)
        self.assertLess(f["straightnessIndex"], 1/math.sqrt(2))

    def test_speed_is_dimensionally_a_length_over_time(self):
        f = SwipeFeaturesV11.extract(self._line(length=100.0, dur=0.1), screen_diag=1000.0)
        self.assertAlmostEqual(f["meanSpeed_px_s"], 1000.0, places=6)

    def test_screen_normalisation_removes_device_scale(self):
        small = SwipeFeaturesV11.extract(self._line(length=100.0), screen_diag=800.0)
        large = SwipeFeaturesV11.extract(
            [(2*x, 2*y, t) for x, y, t in self._line(length=100.0)], screen_diag=1600.0)
        self.assertAlmostEqual(small["snPathLength"], large["snPathLength"], places=9)

    def test_zero_duration_is_rejected(self):
        with self.assertRaises(ValueError):
            SwipeFeaturesV11.extract([(0, 0, 0.0), (10, 0, 0.0)], screen_diag=1000.0)


class TestTapV11(unittest.TestCase):
    def test_tap_rate_is_never_truncated_to_zero(self):
        """The v1.0 defect: 1 ~/ duration is 0 for any duration above 1 ms."""
        f = TapFeaturesV11.extract(0, 80_000, (10, 10), (10, 10), (10, 10), (400, 800))
        self.assertAlmostEqual(f["tapRate_hz"], 12.5, places=9)
        self.assertGreater(f["tapRate_hz"], 0)

    def test_drift_uses_the_y_axis(self):
        """v1.0 subtracted .dx where .dy was meant, so pure vertical drift read as zero."""
        f = TapFeaturesV11.extract(0, 50_000, (100, 100), (100, 106), (100, 100), (400, 800))
        self.assertAlmostEqual(f["withinTapDrift_px"], 6.0, places=9)

    def test_drift_and_targeting_error_are_distinct(self):
        f = TapFeaturesV11.extract(0, 50_000, (120, 100), (123, 104), (100, 100), (400, 800))
        self.assertAlmostEqual(f["withinTapDrift_px"], 5.0, places=9)
        self.assertAlmostEqual(f["targetingError_px"], 20.0, places=9)

    def test_non_positive_duration_is_rejected(self):
        with self.assertRaises(ValueError):
            TapFeaturesV11.extract(1000, 1000, (0, 0), (0, 0), (0, 0), (400, 800))


class TestKeystrokeV11(unittest.TestCase):
    def _ev(self, press_ms, hold_ms, kt="textKey"):
        return {"press_us": int(press_ms * 1000), "release_us": int((press_ms + hold_ms) * 1000),
                "keyText": "a", "keyType": kt}

    def test_negative_hold_times_are_rejected_not_averaged(self):
        v = KeystrokeSessionVectorV11.build(
            [self._ev(0, 100), {"press_us": 500_000, "release_us": 400_000,
                                "keyText": "b", "keyType": "textKey"}, self._ev(1000, 120)])
        self.assertEqual(v["n_rejected"], 1)
        self.assertAlmostEqual(v["hold_ms_median"], 110.0, places=6)

    def test_pauses_are_excluded_from_flight_time(self):
        ev = [self._ev(0, 100), self._ev(200, 100), self._ev(600_000, 100)]
        v = KeystrokeSessionVectorV11.build(ev)
        self.assertLess(v["flight_ms_median"], KeystrokeSessionVectorV11.PAUSE_MS)

    def test_rate_is_not_integer_truncated(self):
        """v1.0 divided by Duration.inSeconds, so any sub-second session divided by zero."""
        v = KeystrokeSessionVectorV11.build([self._ev(0, 50), self._ev(100, 50)])
        self.assertTrue(np.isfinite(v["keyRate_hz"]))
        self.assertGreater(v["keyRate_hz"], 0)

    def test_cv_is_undefined_rather_than_negative(self):
        v = KeystrokeSessionVectorV11.build([self._ev(0, 100)])
        self.assertTrue(np.isnan(v["hold_ms_cv"]) or v["hold_ms_cv"] >= 0)

    def test_empty_session_does_not_raise(self):
        v = KeystrokeSessionVectorV11.build([])
        self.assertEqual(v["n_events"], 0)


class TestGaitV11(unittest.TestCase):
    def test_orientation_uses_three_axes(self):
        """Device flat on a table: gravity along +z, so pitch and roll are ~0."""
        o = GaitFeaturesV11.orientation(0.0, 0.0, 9.81)
        self.assertAlmostEqual(o["pitch"], 0.0, places=6)
        self.assertAlmostEqual(o["roll"], 0.0, places=6)

    def test_orientation_detects_a_90_degree_tilt(self):
        o = GaitFeaturesV11.orientation(9.81, 0.0, 0.0)
        self.assertAlmostEqual(o["pitch"], -90.0, places=4)

    def test_equal_components_do_not_produce_the_v10_constant(self):
        """v1.0 fed three near-equal magnitudes here and got the same angle every time."""
        o = GaitFeaturesV11.orientation(5.66, 5.66, 5.66)
        self.assertNotAlmostEqual(o["pitch"], -19.82, places=1)

    def test_jerk_is_not_structurally_signed(self):
        t = np.arange(0, 1, 0.01)
        f = GaitFeaturesV11.step_features(9.81 + np.sin(2*np.pi*2*t), t)
        self.assertGreater(f["jerkSignBalance"], 0.3)
        self.assertLess(f["jerkSignBalance"], 0.7)

    def test_gravity_is_removed(self):
        f = GaitFeaturesV11.step_features(np.full(50, 9.80665), np.arange(50) * 0.01)
        self.assertAlmostEqual(f["meanLinearAccel_ms2"], 0.0, places=6)


suite = unittest.TestSuite()
for tc in (TestGyroV11, TestSwipeV11, TestTapV11, TestKeystrokeV11, TestGaitV11):
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(tc))
res = unittest.TextTestRunner(verbosity=2).run(suite)

keep("T28_unit_test_summary", pd.DataFrame({
    "Suite": ["GyroPipelineV11", "SwipeFeaturesV11", "TapFeaturesV11",
              "KeystrokeSessionVectorV11", "GaitFeaturesV11", "TOTAL"],
    "Tests": [6, 6, 4, 5, 5, 26],
    "Property under test": [
        "dimensional separation, dt window, integration correctness, axis-direction naming",
        "path vs chord, curvature below the v1.0 floor, unit correctness, screen invariance",
        "rate not integer-truncated, y-axis drift, drift vs targeting error",
        "invalid holds rejected, pauses segmented, rates finite, CV well-defined",
        "three-axis orientation, signed jerk, gravity removal",
        "all properties violated by v1.0 and upheld by v1.1"],
    "Result": ["pass"] * 6}))
print(f"\nran {res.testsRun} tests: failures {len(res.failures)}, errors {len(res.errors)}")

test_constant_rate_integrates_to_correct_angle (__main__.TestGyroV11.test_constant_rate_integrates_to_correct_angle)
1 rad/s about x for 1 s must integrate to 1 rad of cumulative rotation. ... ok
test_dt_window_rejects_out_of_range_samples (__main__.TestGyroV11.test_dt_window_rejects_out_of_range_samples) ... ok
test_no_output_is_produced_below_the_speed_threshold (__main__.TestGyroV11.test_no_output_is_produced_below_the_speed_threshold) ... ok
test_omega_direction_is_not_orientation (__main__.TestGyroV11.test_omega_direction_is_not_orientation)
Documents the v1.0 conflation explicitly (R1-10). ... ok
test_pure_z_rotation_gives_zero_tilt_drift (__main__.TestGyroV11.test_pure_z_rotation_gives_zero_tilt_drift)
Rotating about z alone must not accumulate roll or pitch. ... ok
test_threshold_units_are_separate (__main__.TestGyroV11.test_threshold_units_are_separate)
A sample may pass on angular acceleration while failing on angular speed; ... ok
test_path_length_exceeds_chord_for_a_curve (


ran 26 tests: failures 0, errors 0


## 17. Measured impact of the v1.1 corrections

> Supports **R1-4**: measurable software-quality outcomes rather than a participant count.

We re-run the corrected pipelines over the released v1.0 data wherever the necessary raw
inputs survive, and quantify what changes. Where raw inputs were never stored — swipe
trajectories, accelerometer axes — recomputation is impossible, and we say so rather than
estimate.

In [43]:
# --- 17.1 Gyroscope: data-driven threshold vs the v1.0 constant -----------
g_ok = g.dropna(subset=["_dt"])
g_ok = g_ok[g_ok._dt.between(1e-4, 0.25)]
thr = GyroPipelineV11.calibrate(g_ok._mag, target_retention=0.30)
print("GYROSCOPE THRESHOLD")
print(f"  v1.0 constant                              : 20 rad/s (mixed dimensions)")
print(f"  v1.1 data-driven speed threshold (p70)     : {thr:.3f} rad/s = {np.degrees(thr):.0f} deg/s")
print(f"  retention under v1.0 filter                : {100*(c_mag|c_spd|c_acc).mean():.2f}%")
print(f"  retention under v1.1 speed gate            : {100*(g_ok._mag > thr).mean():.2f}%")
print(f"  samples admitted by the v1.1 dt window     : "
      f"{100*len(g_ok)/len(g):.2f}% (rest have dt outside [0.1 ms, 250 ms])")

# --- 17.2 Tap: corrected features ----------------------------------------
tp = DF["Tap_Data"]
dx = tp.TapFinalGlobalLocationX - tp.TapInitialGlobalLocationX
dy = tp.TapFinalGlobalLocationY - tp.TapInitialGlobalLocationY
drift_fixed = np.hypot(dx, dy)
rate_fixed = 1000.0 / tp.TapDuration.replace(0, np.nan)
print("\nTAP FEATURES")
print(f"  TapSpeed  v1.0: {100*(tp.TapSpeed==0).mean():.2f}% exactly zero, "
      f"{tp.TapSpeed.nunique()} distinct values")
print(f"  tapRate   v1.1: median {rate_fixed.median():.1f} Hz, "
      f"IQR {rate_fixed.quantile(.25):.1f}-{rate_fixed.quantile(.75):.1f}, "
      f"{rate_fixed.round(2).nunique():,} distinct values")
print(f"  tapDrift  v1.0: median {tp.tapDrift.median():.1f} px (min {tp.tapDrift.min():.1f} px)")
print(f"  drift     v1.1: median {drift_fixed.median():.2f} px, "
      f"p75 {drift_fixed.quantile(.75):.2f} px, p95 {drift_fixed.quantile(.95):.1f} px")
print(f"  taps with exactly zero drift (clean press): {100*(drift_fixed==0).mean():.1f}%")

# --- 17.3 Keystroke: v1.1 session vectors from raw events -----------------
def v11_session(dfk):
    ev = [{"press_us": int(p) * 1000, "release_us": int(r) * 1000,
           "keyText": k, "keyType": str(t).replace("KeyTypes.", "")}
          for p, r, k, t in zip(dfk.pressTime, dfk.releaseTime, dfk.keyText, dfk.keyType)
          if np.isfinite(p) and np.isfinite(r)]
    return KeystrokeSessionVectorV11.build(ev)

src = DF["Keystroke_Data_English"]
sample_keys = src._skey.drop_duplicates().sample(min(400, src._skey.nunique()), random_state=0)
v11 = pd.DataFrame([v11_session(d) for _, d in src[src._skey.isin(sample_keys)].groupby("_skey")])
v10 = src[src._skey.isin(sample_keys)].groupby("_skey")[AGG].first()

print(f"\nKEYSTROKE SESSION VECTORS ({len(v11)} sessions recomputed)")
print(f"  v1.0 AverageKeyPressDuration : median {v10.AverageKeyPressDuration.median():.1f} ms, "
      f"max {v10.AverageKeyPressDuration.max():.0f}")
print(f"  v1.1 hold_ms_median          : median {v11.hold_ms_median.median():.1f} ms, "
      f"max {v11.hold_ms_median.max():.0f}")
print(f"  v1.0 AverageInterkeyTime     : median {v10.AverageInterkeyTime.median():.0f} ms, "
      f"max {v10.AverageInterkeyTime.max():.0f}")
print(f"  v1.1 flight_ms_median        : median {v11.flight_ms_median.median():.0f} ms, "
      f"max {v11.flight_ms_median.max():.0f}   <- pause segmentation")
print(f"  key events rejected as implausible by v1.1 : {int(v11.n_rejected.sum())} "
      f"of {int(v11.n_events.sum())} ({100*v11.n_rejected.sum()/max(v11.n_events.sum(),1):.2f}%)")
print(f"  v1.1 features per session vector           : {v11.shape[1]}")

impact = keep("T29_v11_impact", pd.DataFrame({
    "Pipeline": ["Gyroscope filter", "Tap rate", "Tap drift", "Keystroke flight time",
                 "Keystroke session vector"],
    "v1.0 behaviour": [
        f"single scalar 20 across three dimensions; {100*(c_mag|c_spd|c_acc).mean():.1f}% retained",
        f"{100*(tp.TapSpeed==0).mean():.1f}% of values exactly 0 (integer division)",
        f"median {tp.tapDrift.median():.0f} px, floor {tp.tapDrift.min():.0f} px (axis typo)",
        f"max {src.flightTime.max()/60000:.0f} min recorded as one transition",
        "11 scalars, unguarded division, wall-clock timing"],
    "v1.1 behaviour": [
        f"per-dimension thresholds; speed gate {thr:.2f} rad/s, {100*(g_ok._mag>thr).mean():.1f}% retained",
        f"median {rate_fixed.median():.1f} Hz, {rate_fixed.round(2).nunique():,} distinct values",
        f"median {drift_fixed.median():.2f} px, p95 {drift_fixed.quantile(.95):.0f} px",
        f"pauses > {KeystrokeSessionVectorV11.PAUSE_MS:.0f} ms segmented out",
        f"{v11.shape[1]} features, guarded division, monotonic clock"]}))
impact

GYROSCOPE THRESHOLD
  v1.0 constant                              : 20 rad/s (mixed dimensions)
  v1.1 data-driven speed threshold (p70)     : 1.629 rad/s = 93 deg/s
  retention under v1.0 filter                : 98.53%
  retention under v1.1 speed gate            : 30.00%
  samples admitted by the v1.1 dt window     : 87.03% (rest have dt outside [0.1 ms, 250 ms])

TAP FEATURES
  TapSpeed  v1.0: 99.29% exactly zero, 2 distinct values
  tapRate   v1.1: median 14.9 Hz, IQR 11.5-20.4, 283 distinct values
  tapDrift  v1.0: median 420.6 px (min 102.4 px)
  drift     v1.1: median 0.00 px, p75 4.35 px, p95 201.7 px
  taps with exactly zero drift (clean press): 53.5%

KEYSTROKE SESSION VECTORS (400 sessions recomputed)
  v1.0 AverageKeyPressDuration : median 94.3 ms, max 198
  v1.1 hold_ms_median          : median 93.5 ms, max 177
  v1.0 AverageInterkeyTime     : median 1152 ms, max 10757
  v1.1 flight_ms_median        : median 353 ms, max 2200   <- pause segmentation
  key events rejected as 

,Pipeline,v1.0 behaviour,v1.1 behaviour
0,Gyroscope filter,single scalar 20 across three dimensions; 98.5...,per-dimension thresholds; speed gate 1.63 rad/...
1,Tap rate,99.3% of values exactly 0 (integer division),"median 14.9 Hz, 283 distinct values"
2,Tap drift,"median 421 px, floor 102 px (axis typo)","median 0.00 px, p95 202 px"
3,Keystroke flight time,max 37 min recorded as one transition,pauses > 3000 ms segmented out
4,Keystroke session vector,"11 scalars, unguarded division, wall-clock timing","28 features, guarded division, monotonic clock"


In [44]:
fig, ax = plt.subplots(1, 3, figsize=(11.6, 3.2))
ax[0].hist(g_ok._mag, bins=80, color="#c9c9c9", label="all samples")
ax[0].axvline(20, color="#b23b3b", ls="--", lw=1.4, label="v1.0 gate (20 rad/s)")
ax[0].axvline(thr, color="#2e7d4f", ls="-", lw=1.6, label=f"v1.1 gate ({thr:.2f} rad/s)")
ax[0].set_xlabel(r"$|\omega|$ (rad/s)"); ax[0].set_ylabel("records"); ax[0].legend(fontsize=7)
ax[0].set_title("(a) Gyroscope significance gate")

ax[1].hist(np.clip(tp.TapSpeed, 0, 2), bins=[0, .5, 1, 1.5, 2], color="#b23b3b",
           alpha=.75, label="v1.0 TapSpeed")
ax[1].set_xlabel("value"); ax[1].set_ylabel("taps"); ax[1].set_yscale("log")
ax[1].legend(fontsize=7); ax[1].set_title("(b) v1.0 tap rate collapses to 0")

ax[2].hist(rate_fixed.clip(upper=60), bins=60, color="#4f8a6d", label="v1.1 tapRate (Hz)")
ax[2].set_xlabel("taps/s"); ax[2].set_ylabel("taps"); ax[2].legend(fontsize=7)
ax[2].set_title("(c) v1.1 tap rate is informative")
plt.savefig(FIGS / "fig_v11_impact.png", bbox_inches="tight"); plt.show()

### 17.1 Security posture — what v1.0 actually did

> **Addresses R1-6, R1-7** and Reviewer 2's remark that *"the term 'end-to-end encryption'
> may be worth reconsidering"*.

This cannot be answered from the datasets; it is answered from the source, and it must be
answered plainly. The table below is a factual audit of the v1.0 client, with file and line
references, and the corresponding v1.1 remediation. The manuscript's "end-to-end
encryption" claim is **withdrawn in full**.

In [45]:
sec = pd.DataFrame([
    ("Transport", "All 24 endpoints use plain HTTP to a hard-coded IPv4 address "
     "(`http://15.184.243.127:8080/...`)", "no TLS; payloads observable on the path",
     "HTTPS with certificate pinning; host from build-time config"),
    ("Data at rest (device)", "`Hive.openBox('offline<Modality>Data')` opened with no "
     "`encryptionCipher` in all eight offline boxes", "cached payloads stored as plaintext",
     "`HiveAesCipher` with a 256-bit key held in the Android Keystore"),
    ("Authentication", "`authToken` written to `flutter_secure_storage` at login but never "
     "attached to any upload request", "collection endpoints are effectively unauthenticated",
     "bearer token on every request; server-side verification; refresh rotation"),
    ("Backend", "Firebase dependencies are commented out in `pubspec.yaml`; the client "
     "posts to a Dart Frog service on EC2. Only the admin dashboard uses Firestore",
     "the architecture described in the manuscript was not the architecture deployed",
     "single documented backend; architecture figure corrected to match"),
    ("Credentials", "Admin dashboard reads a service-account JSON from an absolute developer "
     "path (`export_view.dart`)", "not reproducible; risks credential leakage",
     "environment variable + documented setup; sample template committed"),
    ("Export", "`export_data.py` includes an `Email` column in exported CSVs",
     "a direct identifier can leave the system", "identifier columns removed; export is "
     "pseudonymous by construction"),
    ("Key custody", "No key generation, storage or rotation of any kind",
     "'end-to-end encryption' was not implemented in any sense",
     "documented key hierarchy; per-install data key wrapped by the Keystore"),
], columns=["Aspect", "v1.0 as implemented (file evidence)", "Consequence", "v1.1 remediation"])
keep("T30_security_audit", sec)
for _, r in sec.iterrows():
    print(f"[{r.Aspect}]\n  v1.0 : {r['v1.0 as implemented (file evidence)']}"
          f"\n  risk : {r.Consequence}\n  v1.1 : {r['v1.1 remediation']}\n")

[Transport]
  v1.0 : All 24 endpoints use plain HTTP to a hard-coded IPv4 address (`http://15.184.243.127:8080/...`)
  risk : no TLS; payloads observable on the path
  v1.1 : HTTPS with certificate pinning; host from build-time config

[Data at rest (device)]
  v1.0 : `Hive.openBox('offline<Modality>Data')` opened with no `encryptionCipher` in all eight offline boxes
  risk : cached payloads stored as plaintext
  v1.1 : `HiveAesCipher` with a 256-bit key held in the Android Keystore

[Authentication]
  v1.0 : `authToken` written to `flutter_secure_storage` at login but never attached to any upload request
  risk : collection endpoints are effectively unauthenticated
  v1.1 : bearer token on every request; server-side verification; refresh rotation

[Backend]
  v1.0 : Firebase dependencies are commented out in `pubspec.yaml`; the client posts to a Dart Frog service on EC2. Only the admin dashboard uses Firestore
  risk : the architecture described in the manuscript was not the architect

In [46]:
threat = pd.DataFrame([
    ("Lost or stolen participant device", "Local Hive cache readable offline",
     "v1.0: none. v1.1: AES-256 Hive cipher, key in Android Keystore, "
     "cache purged on successful upload"),
    ("Intercepted synchronisation traffic", "Plaintext behavioural payloads on the network",
     "v1.0: none. v1.1: TLS 1.3 with certificate pinning"),
    ("Compromised participant account", "Impersonation, submission of forged sessions",
     "v1.1: bearer-token auth with rotation; server-side session-identifier checks"),
    ("Malicious or careless administrator", "Bulk export of the full corpus",
     "v1.1: role-based access control on the dashboard; append-only export audit log"),
    ("Backend database compromise", "Disclosure of the whole corpus",
     "v1.1: encryption at rest; identifiers stored separately from behavioural records"),
    ("Re-identification of pseudonymous records", "Behavioural biometrics are persistent "
     "identifiers", "Minimal demographics (sex, birth year, skill band); no email, phone "
     "or name in any export; identifier columns removed from the export path"),
    ("Replayed or duplicated sessions", "Corpus pollution",
     "v1.1: server-side deduplication on (participantId, sessionId, modality)"),
    ("Unauthorised dataset export", "Data leaves the study",
     "v1.1: exports require an authenticated admin role and are logged"),
], columns=["Threat", "Exposure", "Control"])
keep("T31_threat_model", threat)
threat

,Threat,Exposure,Control
0,Lost or stolen participant device,Local Hive cache readable offline,"v1.0: none. v1.1: AES-256 Hive cipher, key in ..."
1,Intercepted synchronisation traffic,Plaintext behavioural payloads on the network,v1.0: none. v1.1: TLS 1.3 with certificate pin...
2,Compromised participant account,"Impersonation, submission of forged sessions",v1.1: bearer-token auth with rotation; server-...
3,Malicious or careless administrator,Bulk export of the full corpus,v1.1: role-based access control on the dashboa...
4,Backend database compromise,Disclosure of the whole corpus,v1.1: encryption at rest; identifiers stored s...
5,Re-identification of pseudonymous records,Behavioural biometrics are persistent identifiers,"Minimal demographics (sex, birth year, skill b..."
6,Replayed or duplicated sessions,Corpus pollution,v1.1: server-side deduplication on (participan...
7,Unauthorised dataset export,Data leaves the study,v1.1: exports require an authenticated admin r...


## 18. Export of all result tables

In [47]:
for name, df in RESULT_TABLES.items():
    out = df.reset_index() if df.index.name else df
    out.to_csv(RESULTS / f"{name}.csv", index=False)
print(f"wrote {len(RESULT_TABLES)} tables to {RESULTS}/\n")
for n in sorted(RESULT_TABLES): print(" ", n, RESULT_TABLES[n].shape)
print(f"\nfigures in {FIGS}/:")
for f in sorted(FIGS.glob("*.png")): print(" ", f.name)

headline = {
    "enrolled_participants": ENROLLED,
    "contributing_participants": int(len(contributing)),
    "contribution_rate_pct": round(100*len(contributing)/ENROLLED, 1),
    "acquisition_window": f"{allt.min():%Y-%m-%d} to {allt.max():%Y-%m-%d}",
    "acquisition_span_days": int((allt.max()-allt.min()).days),
    "total_records": int(inventory.Records.sum()),
    "total_sessions": int(inventory.Sessions.sum()),
    "keystroke_session_vectors": int(sum(DF[n]['_skey'].nunique() for n in ORDER if n.startswith('Keystroke'))),
    "exported_feature_columns": int(tax["Exported feature columns"].sum()),
    "median_modalities_per_participant": int(per_p.median()),
    "participants_all_11_modalities": int((per_p == 11).sum()),
    "retention_30d_pct": float(retention.loc[retention["Days after first session"] == 30,
                                             "% of contributors (n=265)"].iloc[0]),
    "retention_60d_pct": float(retention.loc[retention["Days after first session"] == 60,
                                             "% of contributors (n=265)"].iloc[0]),
    "median_observation_span_days": int(span.median()),
    "median_sessions_per_participant": int(nsess.median()),
    "device_screen_diag_min_px": int(part_diag.min()),
    "device_screen_diag_max_px": int(part_diag.max()),
    "device_profiles_distinct": int(sess_diag.round(0).nunique()),
    "gyro_records_retained_by_v10_filter_pct": round(100*float((c_mag|c_spd|c_acc).mean()), 2),
    "gyro_retained_by_acceleration_term_only_pct": round(100*float((c_acc & ~c_mag & ~c_spd).mean()), 2),
    "gyro_magnitude_p999_rad_s": round(float(g._mag.quantile(.999)), 2),
    "cross_modal_overlap_pct": round(100*len(involved)/len(IV), 2),
    "amharic_keys_per_char": round(float(kpc_am.median()), 2),
    "english_keys_per_char": round(float(kpc_en.median()), 2),
    "unit_tests_passed": int(res.testsRun - len(res.failures) - len(res.errors)),
    "unit_tests_total": int(res.testsRun),
}
(RESULTS / "headline_numbers.json").write_text(json.dumps(headline, indent=2))
print("\nheadline numbers used in the manuscript and the response letter:")
print(json.dumps(headline, indent=2))

wrote 31 tables to resultsBahri/

  T01_export_integrity (23, 5)
  T02_modality_inventory (11, 10)
  T03_completeness_by_modality (11, 8)
  T04_validity_rule_breakdown (31, 4)
  T05_modality_coverage (11, 2)
  T06_participant_overlap_jaccard (11, 12)
  T07_cross_modality_overlap (14, 2)
  T08_retention_curve (7, 4)
  T09_engagement_summary (3, 5)
  T10_monthly_active (5, 2)
  T11_device_screen_classes (5, 3)
  T12_device_normalisation_effect (4, 3)
  T13_timestamp_resolution_and_jitter (4, 7)
  T14_realised_gyro_rate (10, 2)
  T15_gyro_filter_audit (4, 5)
  T16_gyro_gate_equivalence (5, 2)
  T17_rollpitch_audit (6, 2)
  T18_accelerometer_defects (4, 3)
  T19_swipe_field_status (11, 3)
  T20_tap_field_status (8, 3)
  T21_keystroke_metric_plausibility (11, 6)
  T22_keystroke_redundancy (3, 3)
  T23_amharic_composition (8, 2)
  T24_amharic_vs_english_timing (4, 3)
  T25_handwriting_fidelity (6, 3)
  T26_feature_taxonomy (11, 7)
  T27_registry_reconciliation (4, 4)
  T28_unit_test_summary 

---

## Summary of dispositions

| Finding | Reviewer | Disposition in revision 1 |
|---|---|---|
| Novelty claims unsupported by a structured comparison | R1-1, R2 | Comparison table added with a documented search protocol; "first"/"largest" softened to a bounded, cited claim |
| Eleven modalities not enumerated | R1-2, R2 | §3 table: modality, task, sensor, raw signal, counts, output file |
| "Concurrent" is misleading | R1-3 | §4.1: <1% cross-modal overlap; wording changed to "unified ... within a single application" |
| No empirical software validation | R1-4 | §§2, 3.1, 8–13, 17: export integrity, validity rates, defect audit, v1.1 impact |
| No retention analysis | R1-5, R2 | §5: full survival curve, monthly actives, session distribution; gamification claim softened |
| "End-to-end encryption" | R1-6, R2 | §17.1: claim withdrawn; factual audit with file evidence and v1.1 remediation |
| No threat model | R1-7 | §17.1: eight-threat model mapped to controls |
| Ethics approval not reported | R1-8, R2 | Added to the manuscript (approval body, reference, consent, withdrawal, retention) |
| Gyroscope threshold dimensionally inconsistent | R1-9, R2 | §8: audit; §16.1 corrected per-dimension, data-driven thresholds |
| roll/pitch from angular velocity | R1-10, R2 | §9: columns retracted and renamed; §16.1 complementary filter added |
| Algorithm 1 not reproducible | R1-11, R2 | §7 units contract; §16 executable implementation replaces the pseudocode |
| Device heterogeneity deferred | R1-12 | §6: 71 recovered device profiles, 2.4× spread, normalisation effectiveness measured |
| 101 vs 578 features unexplained | R1-13 | §15: taxonomy reconciling exported columns, session vector and registry |
| Completeness not reported by modality | R1-14 | §3.1: valid/discarded/missing per modality with rule-level breakdown |
| Repository link not version-persistent | R1-15 | v1.0 tag + Zenodo DOI (manuscript metadata) |
| Compilation requirements under-specified | R1-16 | Pinned versions, lockfiles, `.env` template, sample data (repository) |
| Amharic input handling underdescribed | R2 | §14: two-tier fidel model; hold/flight defined per physical key event |
| No unit tests | R2 | §16.3: 26 tests over five extractors |
| Handwriting temporal fidelity overstated | R2 | §14.1: v1.0 preserves space, not time; corrected in v1.1 |